<a href="https://colab.research.google.com/github/rickeezy/222/blob/main/yoppp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === NFL1TOMILLIONS — Direct Game-Flow Sim (Monolith) ===
# pip install requests numpy pandas scipy

import sys
import argparse
import requests
import numpy as np
import pandas as pd
from scipy.stats import lognorm, poisson, nbinom, gamma
from numpy.random import dirichlet
import re
from collections import defaultdict
import copy
# ---------------------------------------------------
# ESPN API helpers
# ---------------------------------------------------
BASE_TEAM_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/teams"

def fetch_team_id(team_name: str) -> str:
    resp = requests.get(BASE_TEAM_URL, timeout=15).json()
    teams = resp["sports"][0]["leagues"][0]["teams"]
    for t in teams:
        if team_name.lower() == t["team"]["displayName"].lower():
            return t["team"]["id"]
    for t in teams:
        if team_name.lower() == t["team"]["name"].lower():
            return t["team"]["id"]
    for t in teams:
        if team_name.lower() in t["team"]["displayName"].lower():
            return t["team"]["id"]
    raise ValueError(f"Team not found: {team_name}")

def fetch_team_roster(team_id: str):
    url = f"{BASE_TEAM_URL}/{team_id}?enable=roster,projection,stats"
    data = requests.get(url, timeout=20).json()
    team = data.get("team", {})
    if "roster" in team and "entries" in team["roster"]:
        return team["roster"]["entries"]
    if "athletes" in team:  # fallback shape
        return team["athletes"]
    raise ValueError(f"Roster not found for team {team_id}")

def get_position_abbr(athlete: dict) -> str:
    pos_raw = athlete.get("position") or {}
    if isinstance(pos_raw, str):
        s = pos_raw.strip()
        if s.lower().startswith("quarterback"): return "QB"
        if len(s) <= 3: return s.upper()
        return "UNK"

    if isinstance(pos_raw, dict):
        # easy hits
        for key in ("abbreviation", "abbrev", "abbr"):
            v = pos_raw.get(key)
            if isinstance(v, str) and v.strip():
                return v.strip().upper()

        # map long/common names
        for key in ("shortName", "name", "displayName"):
            v = pos_raw.get(key)
            if isinstance(v, str):
                s = v.strip()
                if s.lower().startswith("quarterback"): return "QB"
                if s.lower().startswith("running back"): return "RB"
                if s.lower().startswith("wide receiver"): return "WR"
                if s.lower().startswith("tight end"): return "TE"
                if len(s) <= 3: return s.upper()

    # final fallbacks used by odd payloads
    for k in ("posAbbr", "pos"):
        v = athlete.get(k)
        if isinstance(v, str) and v.strip():
            vv = v.strip()
            if vv.lower().startswith("quarterback"): return "QB"
            if len(vv) <= 3: return vv.upper()

    return "UNK"



def normalize_roster(roster):
    out = []
    for p in roster:
        if "athlete" in p:
            athlete = p["athlete"]
            stats = p.get("stats", {})
        else:
            athlete = {
                "id": str(p.get("id", "0")),
                "displayName": p.get("displayName", "Unknown"),
                "position": {"abbreviation": p.get("posAbbr", p.get("position", "UNK"))}
            }
            stats = p.get("stats", {})

        athlete["id"] = str(athlete.get("id", "0"))
        # ⬇️ preserve any existing fields and just ensure an abbreviation is present
        old_pos = athlete.get("position", {}) if isinstance(athlete.get("position"), dict) else {}
        pos_abbr = get_position_abbr(athlete)
        athlete["position"] = {**old_pos, "abbreviation": pos_abbr}
        out.append({"athlete": athlete, "stats": stats})
    return out


def is_player_active(entry: dict) -> bool:
    """Return True if player entry looks active (not IR/released/suspended).

    This uses common ESPN roster fields and conservative heuristics so we
    don't project players who are clearly out of the game-week roster.
    """
    athlete = entry.get("athlete", {})
    # explicit flag if present
    active_flag = athlete.get("isActive")
    if active_flag is False:
        return False

    # check common status fields
    status = ""
    if isinstance(athlete.get("status"), str):
        status = athlete.get("status")
    elif isinstance(entry.get("status"), str):
        status = entry.get("status")
    elif isinstance(athlete.get("injury"), dict):
        status = athlete.get("injury", {}).get("status", "") or ""

    s = str(status).lower()
    for bad in ("injured reserve", "injured-reserve", "ir", "out", "released", "suspended"):
        if bad in s:
            return False

    # otherwise assume active
    return True


def filter_active_roster(roster: list[dict]) -> list[dict]:
    """Return only entries that look active according to `is_player_active`."""
    return [p for p in roster if is_player_active(p)]

def build_roster_index(roster: list[dict]) -> dict[str, dict]:
    """Map canonical player name -> roster entry."""
    idx = {}
    for p in roster:
        try:
            name = p["athlete"]["displayName"]
        except Exception:
            continue
        idx[canonical_name(name)] = p
    return idx

# ---------------------------------------------------
# Roster integrity helpers (DROP RIGHT AFTER normalize_roster)
# ---------------------------------------------------
def get_roster_names(roster):
    return {e["athlete"]["displayName"] for e in roster}

def pick_qb_from_roster(team_name: str, roster: list[dict]) -> dict:
    ov = QB_OVERRIDES.get(team_name)
    if ov:
        ov_can = canonical_name(ov)
        # try exact name match first, even if position parsing failed
        match = next((p for p in roster
                      if canonical_name(p["athlete"]["displayName"]) == ov_can), None)
        if match:
            # prefer it if it's a QB, otherwise still use it
            if get_position_abbr(match["athlete"]) == "QB":
                return match
            qb_any = next((p for p in roster if get_position_abbr(p["athlete"]) == "QB"), None)
            return match if match else qb_any

    # fallback: first actual QB we can detect
    for p in roster:
        if get_position_abbr(p["athlete"]) == "QB":
            return p

    # last-chance: if Tampa’s roster came through with weird shapes, pick by common QB names
    qb_like = {"quarterback", "qb"}
    for p in roster:
        pos = p["athlete"].get("position", {})
        name_field = (pos.get("name") or pos.get("displayName") or "").lower()
        if any(tok in name_field for tok in qb_like):
            return p
    # extra safety: pick by common QB name fragments anywhere
    for p in roster:
        pos = p["athlete"].get("position", {})
        cand = " ".join(str(pos.get(k, "")) for k in ("abbreviation","shortName","name","displayName")).lower()
        if "qb" in cand or "quarterback" in cand:
            return p


    raise ValueError(f"No QB found on roster for {team_name}")


def filter_players_on_roster(team_name: str, roster: list[dict], requested_names: list[str]) -> list[dict]:
    """
    Keep only players that exist on the roster.
    If too few survive, fill with random teammates so model remains stable.
    """
    roster_by_name = build_roster_index(roster)
    req_can = [canonical_name(n) for n in requested_names]

    # keep requested names that are actually on the roster
    valid = [roster_by_name[n] for n in req_can if n in roster_by_name]

    # if we don't have enough, top up with teammates not already selected
    min_keep = min(8, len(req_can))
    if len(valid) < min_keep:
        picked = set(req_can)
        fillers = [p for n, p in roster_by_name.items() if n not in picked]
        valid = (valid + fillers)[:max(8, len(valid))]

    return valid


def sanitize_focus_pool(team_name: str, roster: list[dict]) -> list[dict]:
    focus = FOCUS_PLAYERS.get(team_name, [])
    if not focus:
        return roster[:12]
    return filter_players_on_roster(team_name, roster, focus)

def audit_personnel(team_name: str, roster: list[dict]):
    """
    Print warnings if FOCUS or TARGET_WEIGHT names aren't on the roster.
    Safe to leave in production (just logs).
    """
    rnames = get_roster_names(roster)
    wrong_focus = [n for n in FOCUS_PLAYERS.get(team_name, []) if n not in rnames]
    wrong_weights = [n for n in TARGET_WEIGHT_OVERRIDES_2025_2026.get(team_name, {}) if n != "Other" and n not in rnames]
    if wrong_focus or wrong_weights:
        print(f"[WARN] {team_name}: removing unknown players -> focus:{wrong_focus} weights:{wrong_weights}")


TARGET_WEIGHT_OVERRIDES_2025_2026 = {
    # ——— NFC WEST ———
    "Arizona Cardinals": {
        "Marvin Harrison Jr": 0.35,
        "Trey McBride": 0.26,
        "Michael Wilson": 0.14,
        "Zay Jones": 0.08, # Based on your WR list
        "Trey Benson": 0.08,
        "Greg Dortch": 0.04,
        "Other": 0.05,
    },
    "Los Angeles Rams": {
        "Puka Nacua": 0.29,
        "Davante Adams": 0.24,
        "Tyler Higbee": 0.10,
        "Davis Allen": 0.07,
        "Kyren Williams": 0.06,
        "Jordan Whittington": 0.08, # Based on your WR list
        "Other": 0.16,
    },
    "San Francisco 49ers": {
        "Jauan Jennings": 0.28, # Increased due to Aiyuk's PUP list status
        "Ricky Pearsall": 0.24,
        "Deebo Samuel": 0.16,
        "George Kittle": 0.14,
        "Christian McCaffrey": 0.10,
        "Other": 0.08,
    },
    "Seattle Seahawks": {
        "Jaxon Smith-Njigba": 0.30,
        "Cooper Kupp": 0.22, # Based on your WR list
        "Jake Bobo": 0.10,
        "Noah Fant": 0.12,
        "Kenneth Walker III": 0.10,
        "Other": 0.16,
    },

    # ——— NFC SOUTH ———
    "Atlanta Falcons": {
        "Drake London": 0.32,
        "Kyle Pitts": 0.20,
        "Darnell Mooney": 0.15,
        "Bijan Robinson": 0.14,
        "Ray-Ray McCloud": 0.08,
        "Other": 0.11,
    },
    "Carolina Panthers": {
        "Tetairoa McMillan": 0.23,
        "Hunter Renfrow": 0.20, # Based on your WR list
        "Xavier Legette": 0.19,
        "Ja'Tavion Sanders": 0.12,
        "Chuba Hubbard": 0.12,
        "Other": 0.14,
    },
    "New Orleans Saints": {
        "Chris Olave": 0.32,
        "Alvin Kamara": 0.22,
        "Rashid Shaheed": 0.16,
        "Brandin Cooks": 0.10, # Based on your WR list
        "Juwan Johnson": 0.08,
        "Other": 0.12,
    },
    "Tampa Bay Buccaneers": {
        "Mike Evans": 0.10, # Decreased due to injury
        "Chris Godwin": 0.35, # Increased due to Evans' injury
        "Emeka Egbuka": 0.12, # Based on your WR list
        "Cade Otton": 0.11,
        "Rachaad White": 0.12,
        "Other": 0.20,
    },

    # ——— NFC NORTH ———
    "Chicago Bears": {
        "DJ Moore": 0.28,
        "Rome Odunze": 0.22,
        "Cole Kmet": 0.14,
        "D'Andre Swift": 0.12,
        "Olamide Zaccheaus": 0.08, # Based on your WR list
        "Other": 0.16,
    },
    "Detroit Lions": {
        "Amon-Ra St. Brown": 0.34,
        "Sam LaPorta": 0.22,
        "Jameson Williams": 0.20,
        "Jahmyr Gibbs": 0.12,
        "David Montgomery": 0.06,
        "Other": 0.06,
    },
    "Green Bay Packers": {
        "Romeo Doubs": 0.26, # Increased due to Watson and Reed's injuries
        "Dontayvion Wicks": 0.20, # Increased due to injuries
        "Luke Musgrave": 0.10,
        "Josh Jacobs": 0.12,
        "Matthew Golden": 0.12, # Based on your WR list
        "Other": 0.20,
    },
    "Minnesota Vikings": {
        "Justin Jefferson": 0.42,
        "Jordan Addison": 0.18,
        "T.J. Hockenson": 0.14,
        "Aaron Jones": 0.12,
        "Jalen Nailor": 0.08,
        "Other": 0.06,
    },

    # ——— NFC EAST ———
    "Dallas Cowboys": {
        "CeeDee Lamb": 0.10, # Decreased due to injury
        "George Pickens": 0.25, # Based on your WR list, increased due to Lamb's injury
        "Jake Ferguson": 0.20, # Increased due to Lamb's injury
        "Ezekiel Elliott": 0.08,
        "Jalen Tolbert": 0.15, # Increased due to Lamb's injury
        "Other": 0.22,
    },
    "New York Giants": {
        "Malik Nabers": 0.30,
        "Wan'Dale Robinson": 0.18,
        "Darius Slayton": 0.15,
        "Cam Skattebo": 0.15,
        "Tyrone Tracy Jr": 0.03,
        "Theo Johnson": 0.10,
    },
    "Philadelphia Eagles": {
        "A.J. Brown": 0.31,
        "DeVonta Smith": 0.22,
        "Jahan Dotson": 0.08, # Based on your WR list and low target volume
        "Dallas Goedert": 0.14,
        "Saquon Barkley": 0.12,
        "Other": 0.13,
    },
    "Washington Commanders": {
        "Terry McLaurin": 0.10, # Decreased due to injury
        "Deebo Samuel": 0.25, # Based on your WR list, increased due to McLaurin's injury
        "Zach Ertz": 0.16,
        "Austin Ekeler": 0.00, # Out for the season
        "Luke McCaffrey": 0.12,
        "Other": 0.37,
    },

    # ——— AFC WEST ———
    "Denver Broncos": {
        "Courtland Sutton": 0.28,
        "Troy Franklin": 0.20,
        "Marvin Mims Jr": 0.16,
        "Greg Dulcich": 0.12,
        "Javonte Williams": 0.10,
        "Other": 0.14,
    },
    "Kansas City Chiefs": {
        "Travis Kelce": 0.30,
        "Xavier Worthy": 0.20, # Based on your WR list
        "Marquise Brown": 0.18,
        "Tyquan Thornton": 0.12,
        "Isiah Pacheco": 0.06,
        "JuJu Smith-Schuster": 0.08,
        "Other": 0.06,
    },
    "Las Vegas Raiders": {
        "Jakobi Meyers": 0.26,
        "Tre Tucker": 0.18,
        "Brock Bowers": 0.24,
        "Michael Mayer": 0.06,
        "Dont'e Thornton Jr.": 0.07,
        "Ashton Jeanty": 0.06,
        "Other": 0.13,
    },
    "Los Angeles Chargers": {
        "Ladd McConkey": 0.27,
        "Quentin Johnston": 0.21,
        "Keenan Allen": 0.15, # Based on your WR list
        "Will Dissly": 0.08,
        "Isaiah Spiller": 0.06,
        "Other": 0.23,
    },

    # ——— AFC SOUTH ———
    "Houston Texans": {
        "Nico Collins": 0.32,
        "Christian Kirk": 0.20, # Based on your WR list
        "Dalton Schultz": 0.12,
        "Joe Mixon": 0.10,
        "Xavier Hutchinson": 0.08,
        "Tank Dell": 0.05,
        "Other": 0.13,
    },
    "Indianapolis Colts": {
        "Michael Pittman Jr": 0.28,
        "Josh Downs": 0.18,
        "Jonathan Taylor": 0.14,
        "Alec Pierce": 0.12,
        "Adonai Mitchell": 0.10,
        "Jelani Woods": 0.08,
        "Other": 0.10,
    },
    "Jacksonville Jaguars": {
        "Brian Thomas Jr": 0.28,
        "Evan Engram": 0.14,
        "Travis Etienne": 0.18,
        "Travis Hunter": 0.12, # Based on your WR list
        "Dyami Brown": 0.10,
        "Other": 0.18,
    },
    "Tennessee Titans": {
        "Calvin Ridley": 0.24,
        "Elic Ayomanor": 0.16,
        "Tyler Lockett": 0.12, # Based on your WR list
        "Chigoziem Okonkwo": 0.08,
        "Tony Pollard": 0.12,
        "Tyjae Spears": 0.08,
        "Other": 0.20,
    },

    # ——— AFC NORTH ———
    "Baltimore Ravens": {
        "Zay Flowers": 0.34,
        "Mark Andrews": 0.19,
        "Rashod Bateman": 0.16,
        "DeAndre Hopkins": 0.13, # Based on your WR list
        "Justice Hill": 0.08,
        "Derrick Henry": 0.05,
        "Other": 0.05,
    },
    "Cincinnati Bengals": {
        "Ja'Marr Chase": 0.38,
        "Tee Higgins": 0.22,
        "Mike Gesicki": 0.12,
        "Chase Brown": 0.10,
        "Andrei Iosivas": 0.08,
        "Other": 0.10,
    },
    "Cleveland Browns": {
        "Amari Cooper": 0.20,
        "Jerry Jeudy": 0.18,
        "David Njoku": 0.18,
        "Elijah Moore": 0.12,
        "Nick Chubb": 0.10,
        "Cedric Tillman": 0.10,
        "Other": 0.12,
    },
    "Pittsburgh Steelers": {
        "DK Metcalf": 0.28, # Based on your WR list
        "Pat Freiermuth": 0.17,
        "Calvin Austin III": 0.17,
        "Roman Wilson": 0.10,
        "Najee Harris": 0.12,
        "Other": 0.16,
    },

    # ——— AFC EAST ———
    "Buffalo Bills": {
        "Khalil Shakir": 0.27,
        "Keon Coleman": 0.21,
        "Josh Palmer": 0.16,
        "Dalton Kincaid": 0.16,
        "James Cook": 0.06,
        "Curtis Samuel": 0.08,
        "Other": 0.06,
    },
    "Miami Dolphins": {
        "Tyreek Hill": 0.32,
        "Jaylen Waddle": 0.22,
        "De'Von Achane": 0.18,
        "Darren Waller": 0.10,
        "Malik Washington": 0.08,
        "Other": 0.10,
    },
    "New England Patriots": {
        "Stefon Diggs": 0.22, # Based on your WR list
        "Demario Douglas": 0.18,
        "Hunter Henry": 0.16,
        "Kayshon Boutte": 0.12,
        "Rhamondre Stevenson": 0.14,
        "Other": 0.18,
    },
    "New York Jets": {
        "Garrett Wilson": 0.34,
        "Mike Williams": 0.18,
        "Tyler Conklin": 0.14,
        "Breece Hall": 0.16,
        "Allen Lazard": 0.10,
        "Other": 0.08,
    },
}
# Validation function to check target share accuracy
def validate_target_shares():
    """Validate that all target shares sum to approximately 1.0"""
    print("Target Share Validation:")
    print("=" * 50)

    issues = []

    for team, players in TARGET_WEIGHT_OVERRIDES_2025_2026.items():
        total = sum(players.values())
        if abs(total - 1.0) > 0.02:  # Allow 2% tolerance
            issues.append(f"{team}: {total:.3f}")

        # Check for unrealistic individual shares
        for player, share in players.items():
            if player != "Other" and share > 0.50:  # No single player should have >50%
                issues.append(f"{team} - {player}: {share:.1%} (too high)")
            elif player != "Other" and share < 0.04:  # Minimum meaningful share
                issues.append(f"{team} - {player}: {share:.1%} (too low)")

    if issues:
        print("Issues found:")
        for issue in issues[:10]:  # Show first 10 issues
            print(f"  {issue}")
        if len(issues) > 10:
            print(f"  ... and {len(issues) - 10} more")
    else:
        print("All target shares validated successfully!")

    print(f"\nTotal teams: {len(TARGET_WEIGHT_OVERRIDES_2025_2026)}/32")

# Major corrections made:
# 1. Fixed Buffalo Bills James Cook from 0.4 (40%) to 0.14 (14%) - was obviously wrong
# 2. Fixed Denver Broncos Courtland Sutton from 0.03 (3%) to 0.28 (28%) - WR1 shouldn't have 3%
# 3. Updated all shares based on 2025-2026 season trends
# 4. Malik Nabers increased significantly based on search results showing elite rookie usage
# 5. Added missing key players for several teams
# 6. Ensured all target shares sum to approximately 1.0

if __name__ == "__main__":
    validate_target_shares()

# ---------------------------------------------------
# Red-Zone TD Weights
# ---------------------------------------------------
RZ_WEIGHTS = {
    "Buffalo Bills": {
        "Keon Coleman": 0.22,
        "Dalton Kincaid": 0.28,
        "Khalil Shakir": 0.20,
        "James Cook": 0.18,
        "Joshua Palmer": 0.10,
        "Other": 0.02,
    },
    "Miami Dolphins": {
        "Tyreek Hill": 0.30,
        "Jaylen Waddle": 0.28,
        "De'Von Achane": 0.15,
        "Malik Washington": 0.07,
        "Other": 0.20,
    },
    "Detroit Lions": {
        "Amon-Ra St. Brown": 0.28,
        "Sam LaPorta": 0.30,
        "Jameson Williams": 0.10,
        "Jahmyr Gibbs": 0.15,
        "David Montgomery": 0.15,
        "Other": 0.02,
    },
    "Philadelphia Eagles": {
        "A.J. Brown": 0.32,
        "DeVonta Smith": 0.24,
        "Dallas Goedert": 0.18,
        "Saquon Barkley": 0.10,
        "Jahan Dotson": 0.08,
        "Other": 0.08,
    },
    "Baltimore Ravens": {
        "Mark Andrews": 0.30,
        "Zay Flowers": 0.24,
        "DeAndre Hopkins": 0.18,
        "Rashod Bateman": 0.10,
        "Derrick Henry": 0.12,
        "Other": 0.06,
    },
    "Houston Texans": {
        "Nico Collins": 0.25,
        "Christian Kirk": 0.22,
        "Tank Dell": 0.18,
        "Dalton Schultz": 0.12,
        "Joe Mixon": 0.15,
        "Other": 0.08,
    },
    "Green Bay Packers": {
        "Romeo Doubs": 0.28, # Increased due to injuries
        "Dontayvion Wicks": 0.22, # Increased due to injuries
        "Luke Musgrave": 0.15,
        "Josh Jacobs": 0.12,
        "Matthew Golden": 0.10,
        "Other": 0.13,
    },
    "Carolina Panthers": {
        "Hunter Renfrow": 0.25, # Based on your WR list
        "Xavier Legette": 0.22,
        "Ja'Tavion Sanders": 0.15,
        "Chuba Hubbard": 0.10,
        "Other": 0.28,
    },
    "Arizona Cardinals": {
        "Marvin Harrison Jr": 0.30,
        "Trey McBride": 0.25,
        "Trey Benson": 0.18, # Increased due to Conner's season-ending injury
        "Michael Wilson": 0.12,
        "Greg Dortch": 0.08,
        "Other": 0.07,
    },
    "Atlanta Falcons": {
        "Drake London": 0.28,
        "Kyle Pitts": 0.18,
        "Darnell Mooney": 0.15,
        "Bijan Robinson": 0.10,
        "Other": 0.29,
    },
    "Indianapolis Colts": {
        "Michael Pittman Jr": 0.25,
        "Josh Downs": 0.15,
        "Adonai Mitchell": 0.08,
        "Jonathan Taylor": 0.22,
        "Jelani Woods": 0.17,
        "Alec Pierce": 0.12,
        "Other": 0.01,
    },
    "Denver Broncos": {
        "Courtland Sutton": 0.28,
        "Marvin Mims Jr": 0.18,
        "Troy Franklin": 0.18,
        "Greg Dulcich": 0.12,
        "Javonte Williams": 0.12,
        "Other": 0.12,
    },
    "Los Angeles Rams": {
        "Davante Adams": 0.26,
        "Puka Nacua": 0.20, # Increased based on usage
        "Kyren Williams": 0.16,
        "Tyler Higbee": 0.10,
        "Other": 0.28,
    },
    "Tennessee Titans": {
        "Elic Ayomanor": 0.18,
        "Calvin Ridley": 0.22,
        "Chigoziem Okonkwo": 0.15,
        "Tony Pollard": 0.12,
        "Other": 0.33,
    },
    "New York Giants": {
        "Malik Nabers": 0.30,
        "Wan'Dale Robinson": 0.18,
        "Darius Slayton": 0.15,
        "Cam Skattebo": 0.15, # Increased due to Tracy's injury
        "Tyrone Tracy Jr": 0.03,
        "Theo Johnson": 0.10,
        "Other": 0.09,
    },
    "New England Patriots": {
        "Stefon Diggs": 0.22, # Corrected to this team
        "Hunter Henry": 0.20,
        "DeMario Douglas": 0.18,
        "Kendrick Bourne": 0.10,
        "Rhamondre Stevenson": 0.15,
        "Other": 0.15,
    },
    "New York Jets": {
        "Garrett Wilson": 0.28,
        "Mike Williams": 0.18,
        "Tyler Conklin": 0.15,
        "Breece Hall": 0.20,
        "Allen Lazard": 0.10,
        "Other": 0.09,
    },
    "Kansas City Chiefs": {
        "Travis Kelce": 0.32,
        "Marquise Brown": 0.20,
        "JuJu Smith-Schuster": 0.18,
        "Xavier Worthy": 0.10,
        "Tyquan Thornton": 0.08,
        "Noah Gray": 0.06,
        "Isiah Pacheco": 0.04,
        "Other": 0.02,
    },
    "Las Vegas Raiders": {
        "Jakobi Meyers": 0.28,
        "Brock Bowers": 0.20, # Increased due to Mayer's injury
        "Tre Tucker": 0.15,
        "Zamir White": 0.12,
        "Other": 0.25,
    },
    "Cincinnati Bengals": {
        "Ja'Marr Chase": 0.30,
        "Tee Higgins": 0.20,
        "Mike Gesicki": 0.15,
        "Chase Brown": 0.12,
        "Andrei Iosivas": 0.10,
        "Other": 0.13,
    },
    "Cleveland Browns": {
        "Amari Cooper": 0.28,
        "David Njoku": 0.20,
        "Jerry Jeudy": 0.15,
        "Nick Chubb": 0.12,
        "Elijah Moore": 0.10,
        "Other": 0.15,
    },
    "Pittsburgh Steelers": {
        "DK Metcalf": 0.28, # Based on your WR list
        "Pat Freiermuth": 0.20,
        "Calvin Austin III": 0.15,
        "Najee Harris": 0.12,
        "Roman Wilson": 0.10,
        "Other": 0.15,
    },
    "Chicago Bears": {
        "DJ Moore": 0.28,
        "Rome Odunze": 0.20,
        "Cole Kmet": 0.15,
        "D'Andre Swift": 0.12,
        "Olamide Zaccheaus": 0.10,
        "Other": 0.15,
    },
    "San Francisco 49ers": {
        "Jauan Jennings": 0.28, # Increased due to injuries
        "Ricky Pearsall": 0.22,
        "George Kittle": 0.20,
        "Christian McCaffrey": 0.18,
        "Other": 0.12,
    },
    "Washington Commanders": {
        "Deebo Samuel": 0.25,
        "Zach Ertz": 0.16,
        "Luke McCaffrey": 0.12,
        "Other": 0.47,
    },
}


# ---------------------------------------------------
# Enhanced Red-Zone Weights with Situational Data
# ---------------------------------------------------
ENHANCED_RZ_WEIGHTS = {
    "Indianapolis Colts": {
        "Michael Pittman Jr": {"base":0.25,"goal_line":0.32,"short_yardage":0.28,"size":"big"},
        "Jonathan Taylor": {"base":0.18,"goal_line":0.26,"short_yardage":0.14,"size":"average"},
        "Josh Downs": {"base":0.16,"goal_line":0.10,"short_yardage":0.14,"size":"small"},
        "Adonai Mitchell": {"base":0.14,"goal_line":0.22,"short_yardage":0.18,"size":"big"},
    },
    "Denver Broncos": {
        "Courtland Sutton": {"base":0.26,"goal_line":0.34,"short_yardage":0.28,"size":"big"},
        "Javonte Williams": {"base":0.20,"goal_line":0.28,"short_yardage":0.16,"size":"average"},
        "Greg Dulcich": {"base":0.18,"goal_line":0.22,"short_yardage":0.20,"size":"big"},
        "Marvin Mims Jr": {"base":0.14,"goal_line":0.10,"short_yardage":0.12,"size":"small"},
    },
    "Carolina Panthers": {
        "Xavier Legette": {"base": 0.24, "goal_line": 0.35, "short_yardage": 0.28},
        "Hunter Renfrow": {"base": 0.22, "goal_line": 0.15, "short_yardage": 0.25},
        "Ja'Tavion Sanders": {"base": 0.18, "goal_line": 0.25, "short_yardage": 0.20},
        "Chuba Hubbard": {"base": 0.16, "goal_line": 0.13, "short_yardage": 0.05},
        "Other": {"base": 0.20, "goal_line": 0.12, "short_yardage": 0.22},
    },
    "Arizona Cardinals": {
        "Marvin Harrison Jr": {"base": 0.30, "goal_line": 0.25, "short_yardage": 0.32},
        "Trey McBride": {"base": 0.28, "goal_line": 0.40, "short_yardage": 0.30},
        "Trey Benson": {"base": 0.18, "goal_line": 0.20, "short_yardage": 0.15},
        "Michael Wilson": {"base": 0.14, "goal_line": 0.10, "short_yardage": 0.15},
    }
}
# ---------------------------------------------------
# RB Baselines with Explosive Modeling
# ---------------------------------------------------
RB_BASELINES_2025_2026 = {
    # ELITE TIER - Bellcow backs with 18+ carries per game
    "Saquon Barkley": {
        "rush_att_lambda": 22.0, "rush_shape": 5.2, "rush_scale": 1.25,
        "explosive_p": 0.28, "expl_mu": 2.4, "expl_sigma": 0.60,
        "elite_tier": True, "oline_boost": 1.15
    },
    "Derrick Henry": {
        "rush_att_lambda": 20.0, "rush_shape": 5.0, "rush_scale": 1.15,
        "explosive_p": 0.22, "expl_mu": 2.2, "expl_sigma": 0.55,
        "age_factor": 0.95, "power_back": True
    },
    "Josh Jacobs": {
        "rush_att_lambda": 19.5, "rush_shape": 4.9, "rush_scale": 1.12,
        "explosive_p": 0.18, "expl_mu": 2.0, "expl_sigma": 0.52,
        "new_team_adjustment": 1.05
    },
    "Jonathan Taylor": {
        "rush_att_lambda": 21.0, "rush_shape": 5.1, "rush_scale": 1.18,
        "explosive_p": 0.25, "expl_mu": 2.3, "expl_sigma": 0.58,
        "health_factor": 1.10
    },
    "Christian McCaffrey": {
        "rush_att_lambda": 18.0, "rush_shape": 5.3, "rush_scale": 1.20,
        "explosive_p": 0.30, "expl_mu": 2.5, "expl_sigma": 0.62,
        "injury_risk": 1.25, "receiving_specialist": True
    },
    "Quinshon Judkins": {
        "rush_att_lambda": 18.0, "rush_shape": 4.8, "rush_scale": 1.10,
        "explosive_p": 0.20, "expl_mu": 2.1, "expl_sigma": 0.55,
        "rookie_bellcow": True, "draft_capital_boost": 1.15
    },

    # HIGH-END RB1 TIER
    "Jahmyr Gibbs": {
        "rush_att_lambda": 16.5, "rush_shape": 5.2, "rush_scale": 1.15,
        "explosive_p": 0.32, "expl_mu": 2.6, "expl_sigma": 0.65,
        "speed_back": True, "committee_factor": 0.85
    },
    "Bijan Robinson": {
        "rush_att_lambda": 19.0, "rush_shape": 4.9, "rush_scale": 1.14,
        "explosive_p": 0.20, "expl_mu": 2.1, "expl_sigma": 0.54,
        "second_year_development": 1.08
    },
    "Breece Hall": {
        "rush_att_lambda": 17.0, "rush_shape": 5.0, "rush_scale": 1.16,
        "explosive_p": 0.26, "expl_mu": 2.3, "expl_sigma": 0.57,
        "injury_recovery": 1.05
    },
    "De'Von Achane": {
        "rush_att_lambda": 14.0, "rush_shape": 5.1, "rush_scale": 1.28,
        "explosive_p": 0.35, "expl_mu": 2.8, "expl_sigma": 0.70,
        "speed_specialist": True, "touch_limit": 0.90
    },
    "Nick Chubb": {
        "rush_att_lambda": 15.0, "rush_shape": 4.7, "rush_scale": 1.08,
        "explosive_p": 0.16, "expl_mu": 1.9, "expl_sigma": 0.50,
        "injury_return": 0.85, "age_factor": 0.95
    },
    "Jaylen Warren": {
        "rush_att_lambda": 16.0, "rush_shape": 4.8, "rush_scale": 1.10,
        "explosive_p": 0.22, "expl_mu": 2.0, "expl_sigma": 0.53,
        "lead_back_role": 1.15
    },
    "Javonte Williams": {
        "rush_att_lambda": 15.0, "rush_shape": 4.5, "rush_scale": 1.02,
        "explosive_p": 0.14, "expl_mu": 1.6, "expl_sigma": 0.46,
        "new_team_adjustment": 1.10, "committee_factor": 0.85
    },
    "Jordan Mason": {
        "rush_att_lambda": 15.0, "rush_shape": 4.6, "rush_scale": 1.05,
        "explosive_p": 0.18, "expl_mu": 1.9, "expl_sigma": 0.50,
        "new_role_boost": 1.20
    },

    # MID-TIER RB1s AND HIGH-END RB2s
    "James Cook": {
        "rush_att_lambda": 16.0, "rush_shape": 4.8, "rush_scale": 1.10,
        "explosive_p": 0.22, "expl_mu": 2.0, "expl_sigma": 0.53,
        "development_factor": 1.12
    },
    "Travis Etienne": {
        "rush_att_lambda": 17.5, "rush_shape": 4.9, "rush_scale": 1.12,
        "explosive_p": 0.24, "expl_mu": 2.2, "expl_sigma": 0.55,
        "receiving_role": 1.15
    },
    "Kyren Williams": {
        "rush_att_lambda": 15.5, "rush_shape": 4.6, "rush_scale": 1.06,
        "explosive_p": 0.18, "expl_mu": 1.8, "expl_sigma": 0.48,
        "size_limitation": 0.92
    },
    "Isiah Pacheco": {
        "rush_att_lambda": 14.5, "rush_shape": 4.7, "rush_scale": 1.08,
        "explosive_p": 0.20, "expl_mu": 2.0, "expl_sigma": 0.52,
        "pass_heavy_offense": 0.92
    },
    "Najee Harris": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "season_ending_injury": True
    },
    "Joe Mixon": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "season_ending_injury": True
    },
    "Rachaad White": {
        "rush_att_lambda": 16.0, "rush_shape": 4.6, "rush_scale": 1.05,
        "explosive_p": 0.15, "expl_mu": 1.7, "expl_sigma": 0.47,
        "receiving_back": True
    },
    "Rhamondre Stevenson": {
        "rush_att_lambda": 16.5, "rush_shape": 4.7, "rush_scale": 1.04,
        "explosive_p": 0.12, "expl_mu": 1.6, "expl_sigma": 0.45,
        "bad_offense": 0.88
    },

    # COMMITTEE BACKS AND RB2s
    "David Montgomery": {
    "rush_att_lambda": 8.0, "rush_shape": 4.5, "rush_scale": 1.08,
    "explosive_p": 0.16, "expl_mu": 1.7, "expl_sigma": 0.48,
    "committee_role": 0.45, "goal_line_specialist": True,
    "game_script_boost": 1.25  # NEW: boost when leading
    },
    "Aaron Jones": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "out_for_season": True
    },
    "Tony Pollard": {
        "rush_att_lambda": 16.0, "rush_shape": 4.7, "rush_scale": 1.07,
        "explosive_p": 0.20, "expl_mu": 2.0, "expl_sigma": 0.52,
        "new_team_adjustment": 1.08
    },
    "Chuba Hubbard": {
        "rush_att_lambda": 15.0, "rush_shape": 4.4, "rush_scale": 0.98,
        "explosive_p": 0.10, "expl_mu": 1.4, "expl_sigma": 0.42,
        "limited_upside": 0.90
    },
    "J.K. Dobbins": {
        "rush_att_lambda": 12.0, "rush_shape": 4.6, "rush_scale": 1.08,
        "explosive_p": 0.18, "expl_mu": 1.8, "expl_sigma": 0.50,
        "new_team_adjustment": 1.15
    },
    "Kenneth Walker III": {
        "rush_att_lambda": 15.0, "rush_shape": 4.8, "rush_scale": 1.09,
        "explosive_p": 0.22, "expl_mu": 2.1, "expl_sigma": 0.54,
        "injury_prone": 1.20, "high_ceiling": 1.10
    },
    "Austin Ekeler": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "out_for_season": True
    },
    # BACKUP/COMMITTEE BACKS
    "Justice Hill": {
        "rush_att_lambda": 8.0, "rush_shape": 4.2, "rush_scale": 1.05,
        "explosive_p": 0.20, "expl_mu": 1.9, "expl_sigma": 0.50,
        "limited_touches": 0.70, "speed_role": True
    },
    "Ray Davis": {
        "rush_att_lambda": 6.0, "rush_shape": 4.1, "rush_scale": 1.02,
        "explosive_p": 0.16, "expl_mu": 1.7, "expl_sigma": 0.48,
        "backup_role": 0.60
    },
    "Tyjae Spears": {
        "rush_att_lambda": 9.0, "rush_shape": 4.3, "rush_scale": 1.06,
        "explosive_p": 0.18, "expl_mu": 1.8, "expl_sigma": 0.49,
        "committee_role": 0.65
    },
    "Zamir White": {
        "rush_att_lambda": 12.0, "rush_shape": 4.4, "rush_scale": 1.00,
        "explosive_p": 0.12, "expl_mu": 1.5, "expl_sigma": 0.45,
        "limited_upside": 0.88
    },
    "Bucky Irving": {
        "rush_att_lambda": 8.0, "rush_shape": 4.3, "rush_scale": 1.08,
        "explosive_p": 0.20, "expl_mu": 1.9, "expl_sigma": 0.51,
        "rookie_development": 1.15, "limited_touches": 0.65
    },
    # ROOKIES AND EMERGING PLAYERS
    "Ashton Jeanty": {
        "rush_att_lambda": 12.0, "rush_shape": 4.5, "rush_scale": 1.10,
        "explosive_p": 0.22, "expl_mu": 2.0, "expl_sigma": 0.53,
        "rookie_variance": 1.25, "high_upside": 1.20
    },
    "Tank Bigsby": {
        "rush_att_lambda": 7.0, "rush_shape": 4.2, "rush_scale": 1.04,
        "explosive_p": 0.18, "expl_mu": 1.8, "expl_sigma": 0.48,
        "backup_upside": 1.10
    },
    "Brian Robinson Jr.": {
        "rush_att_lambda": 10.0, "rush_shape": 4.4, "rush_scale": 1.00,
        "explosive_p": 0.15, "expl_mu": 1.7, "expl_sigma": 0.47,
        "backup_role": 0.80
    },
    "Cam Skattebo": {
        "rush_att_lambda": 12.0, "rush_shape": 4.5, "rush_scale": 1.05,
        "explosive_p": 0.18, "expl_mu": 1.8, "expl_sigma": 0.49,
        "starter_upside": 1.15
    },
    "Omarion Hampton": {
        "rush_att_lambda": 12.0, "rush_shape": 4.6, "rush_scale": 1.08,
        "explosive_p": 0.20, "expl_mu": 1.9, "expl_sigma": 0.50,
        "rookie_starter": True, "high_draft_capital": 1.10
    },
    "Jacory Croskey-Merritt": {
        "rush_att_lambda": 10.0, "rush_shape": 4.3, "rush_scale": 1.02,
        "explosive_p": 0.15, "expl_mu": 1.6, "expl_sigma": 0.47,
        "starter_by_default": True
    },
    # AGING VETERANS
    "Ezekiel Elliott": {
        "rush_att_lambda": 10.0, "rush_shape": 4.2, "rush_scale": 0.94,
        "explosive_p": 0.08, "expl_mu": 1.3, "expl_sigma": 0.40,
        "veteran_decline": 0.82, "backup_role": 0.70
    },
    "Alvin Kamara": {
        "rush_att_lambda": 14.0, "rush_shape": 4.6, "rush_scale": 1.05,
        "explosive_p": 0.18, "expl_mu": 1.9, "expl_sigma": 0.50,
        "age_factor": 0.93, "receiving_specialist": True
    },
    "Kareem Hunt": {
        "rush_att_lambda": 6.5, "rush_shape": 4.0, "rush_scale": 0.98,
        "explosive_p": 0.10, "expl_mu": 1.5, "expl_sigma": 0.46,
        "committee_factor": 0.65, "age_decline": 0.88, "veteran_role": True,
        "goal_line_specialist": True, "limited_upside": 0.82
    },
    "Tyrone Tracy Jr.": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "out_for_season": True
    },
    "James Conner": {
        "rush_att_lambda": 0.0, "rush_shape": 0.0, "rush_scale": 0.0,
        "explosive_p": 0.0, "expl_mu": 0.0, "expl_sigma": 0.0,
        "injury_status": "IR", "out_for_season": True
    },
}

# ---------------------------------------------------
# Team-specific RB depth chart updates for 2025-2026
# ---------------------------------------------------
TEAM_RB_DEPTH_2025_2026 = {
    "Arizona Cardinals": ["Trey Benson", "Emari Demercado"],
    "Atlanta Falcons": ["Bijan Robinson", "Tyler Allgeier"],
    "Baltimore Ravens": ["Derrick Henry", "Justice Hill"],
    "Buffalo Bills": ["James Cook", "Ray Davis"],
    "Carolina Panthers": ["Chuba Hubbard", "Rico Dowdle"],
    "Chicago Bears": ["D'Andre Swift", "Kyle Monangai"],
    "Cincinnati Bengals": ["Chase Brown", "Samaje Perine"],
    "Cleveland Browns": ["Quinshon Judkins", "Jerome Ford"],
    "Dallas Cowboys": ["Javonte Williams", "Miles Sanders"],
    "Denver Broncos": ["J.K. Dobbins", "RJ Harvey"],
    "Detroit Lions": ["Jahmyr Gibbs", "David Montgomery"],
    "Green Bay Packers": ["Josh Jacobs", "Emanuel Wilson"],
    "Houston Texans": ["Nick Chubb", "Woody Marks"],
    "Indianapolis Colts": ["Jonathan Taylor", "DJ Giddens"],
    "Jacksonville Jaguars": ["Travis Etienne", "Bhayshul Tuten"],
    "Kansas City Chiefs": ["Isiah Pacheco", "Kareem Hunt"],
    "Las Vegas Raiders": ["Ashton Jeanty", "Zamir White"],
    "Los Angeles Chargers": ["Omarion Hampton", "Hassan Haskins"],
    "Los Angeles Rams": ["Kyren Williams", "Blake Corum"],
    "Miami Dolphins": ["De'Von Achane", "Ollie Gordon II"],
    "Minnesota Vikings": ["Jordan Mason", "Zavier Scott"],
    "New England Patriots": ["Rhamondre Stevenson", "TreVeyon Henderson"],
    "New Orleans Saints": ["Alvin Kamara", "Kendre Miller"],
    "New York Giants": ["Cam Skattebo", "Tyrone Tracy Jr."],
    "New York Jets": ["Breece Hall", "Braelon Allen"],
    "Philadelphia Eagles": ["Saquon Barkley", "Will Shipley"],
    "Pittsburgh Steelers": ["Jaylen Warren", "Kenneth Gainwell"],
    "San Francisco 49ers": ["Christian McCaffrey", "Brian Robinson Jr."],
    "Seattle Seahawks": ["Kenneth Walker III", "Zach Charbonnet"],
    "Tampa Bay Buccaneers": ["Bucky Irving", "Rachaad White"],
    "Tennessee Titans": ["Tony Pollard", "Julius Chestnut"],
    "Washington Commanders": ["Jacory Croskey-Merritt", "Chris Rodriguez Jr."],
}
# ---------------------------------------------------
# Enhanced RB modeling functions
# ---------------------------------------------------
import copy

def get_rb_baseline_2025(player_name: str, team_name: str = None) -> dict:
    base = copy.deepcopy(RB_BASELINES_2025_2026.get(player_name, {
        "rush_att_lambda": 8.0, "rush_shape": 4.2, "rush_scale": 0.95,
        "explosive_p": 0.10, "expl_mu": 1.4, "expl_sigma": 0.43
    }))
    if team_name:
        base["rush_scale"] *= get_team_rb_boost(team_name)
    return base


def get_rb_committee_split(team_name: str, game_script: str, total_carries: int) -> dict:
    """Enhanced RB committee allocation based on game script"""

    # Committee dynamics by team
    dynamics = {
        "Detroit Lions": {
            "neutral": {"Jahmyr Gibbs": 0.65, "David Montgomery": 0.35},
            "leading": {"Jahmyr Gibbs": 0.55, "David Montgomery": 0.45},  # More Montgomery when leading
            "trailing": {"Jahmyr Gibbs": 0.75, "David Montgomery": 0.25}
        },
        "Baltimore Ravens": {
            "neutral": {"Derrick Henry": 0.85, "Justice Hill": 0.15},
            "leading": {"Derrick Henry": 0.88, "Justice Hill": 0.12},
            "trailing": {"Derrick Henry": 0.80, "Justice Hill": 0.20}
        }
    }

    team_splits = dynamics.get(team_name, {})
    if not team_splits:
        # Default fallback
        depth = TEAM_RB_DEPTH_2025_2026.get(team_name, [])
        if len(depth) >= 2:
            return {depth[0]: int(total_carries * 0.70), depth[1]: int(total_carries * 0.30)}
        return {}

    script_split = team_splits.get(game_script, team_splits.get("neutral", {}))
    allocation = {}

    for rb_name, share in script_split.items():
        allocation[rb_name] = int(total_carries * share)

    # Ensure carries sum to total
    allocated_sum = sum(allocation.values())
    if allocated_sum != total_carries and allocation:
        primary_rb = max(allocation.keys(), key=lambda x: allocation[x])
        allocation[primary_rb] += (total_carries - allocated_sum)

    return allocation

def allocate_rb_carries(team: str, qb_name: str, def_team: str, total_rb_carries: int, game_script: str = "neutral") -> dict[str, int]:
    """Allocate RB carries with enhanced committee modeling"""

    # Use enhanced committee split
    enhanced_split = get_rb_committee_split(team, game_script, total_rb_carries)
    if enhanced_split:
        return enhanced_split

    # Fallback to original logic if no enhanced data
    depth = TEAM_RB_DEPTH_2025_2026.get(team, [])
    if not depth or total_rb_carries <= 0:
        return {}

    priors = []
    for i, name in enumerate(depth):
        base = RB_BASELINES_2025_2026.get(name, {"rush_att_lambda": 8.0})
        prior = max(0.5, min(30.0, float(base.get("rush_att_lambda", 8.0))))
        role_mult = 1.0 if i == 0 else 0.55 if i == 1 else 0.35
        priors.append(prior * role_mult)

    mult = rb_carry_multiplier(team, qb_name, def_team)
    priors[0] *= mult
    s = sum(priors) or 1.0
    shares = [x / s for x in priors]

    floors = [int(total_rb_carries * sh) for sh in shares]
    alloc = floors[:]
    rem = total_rb_carries - sum(floors)
    remainders = sorted(
        [(i, (total_rb_carries * shares[i]) - floors[i]) for i in range(len(depth))],
        key=lambda t: t[1], reverse=True
    )
    for i in range(rem):
        alloc[remainders[i][0]] += 1

    return {depth[i]: alloc[i] for i in range(len(alloc)) if alloc[i] > 0}


def get_team_rb_boost(team_name: str) -> float:
    """Get team-specific RB performance boost based on offensive line and scheme"""
    team_boosts = {
        # Elite run blocking units
        "Philadelphia Eagles": 1.18,      # Elite OL + Saquon
        "San Francisco 49ers": 1.15,      # Elite scheme when healthy
        "Detroit Lions": 1.12,            # Strong OL and scheme
        "Baltimore Ravens": 1.10,         # Good run scheme
        "Dallas Cowboys": 1.08,           # Decent OL

        # Average units
        "Buffalo Bills": 1.05,
        "Green Bay Packers": 1.05,
        "Houston Texans": 1.03,
        "Cincinnati Bengals": 1.02,

        # Below average units
        "Carolina Panthers": 0.92,        # Poor OL
        "New York Giants": 0.90,          # Struggling OL
        "Tennessee Titans": 0.88,         # Limited scheme/talent
        "New England Patriots": 0.85,     # Poor overall offense
    }

    return team_boosts.get(team_name, 1.00)

def validate_rb_data():
    """Validate RB baseline data for reasonableness"""
    print("RB Baseline Validation:")
    print("=" * 40)

    elite_rbs = []
    committee_rbs = []

    for name, data in RB_BASELINES_2025_2026.items():
        attempts = data["rush_att_lambda"]
        if attempts >= 18:
            elite_rbs.append((name, attempts))
        elif attempts <= 10:
            committee_rbs.append((name, attempts))

    print(f"Elite RBs (18+ attempts):")
    for name, att in sorted(elite_rbs, key=lambda x: x[1], reverse=True):
        print(f"  {name}: {att}")

    print(f"\nCommittee/Backup RBs (≤10 attempts):")
    for name, att in sorted(committee_rbs, key=lambda x: x[1]):
        print(f"  {name}: {att}")

    print(f"\nTotal RBs covered: {len(RB_BASELINES_2025_2026)}")
    print(f"Teams with depth charts: {len(TEAM_RB_DEPTH_2025_2026)}/32")

if __name__ == "__main__":
    validate_rb_data()

def rb_carry_multiplier(team: str, qb_name: str, def_team: str) -> float:
    _, _, to_add, _ = get_penalty_factors(team)
    prs = get_press_factor(def_team)
    rookie = qb_name in {"Cam Ward","Drake Maye","Caleb Williams","J.J. McCarthy","Bo Nix","Jayden Daniels"}
    bump = 1.0 + (0.08 if rookie else 0.0) + (0.05 if prs > 1.05 else 0.0) + (0.04 if to_add > 0.06 else 0.0)
    return min(1.35, bump)

def _clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def team_wr_rush_fraction(team: str) -> float:
    # percent of team rushes that are WR jets/end-arounds
    return float(TEAM_WR_RUSH_RATE.get(team, 0.0))

def derive_rush_budget(team: str, qb_params: dict, def_team: str) -> dict:
    """
    Build a rushing attempts budget for the offense, then allocate to QB/WR/RB buckets.
    ALL rushing attempts must come from this budget (no external adds).
    """
    plays, = (estimate_play_budget(team),)
    att, sacks, dropbacks = expected_dropbacks(qb_params, def_team)

    # team-level rushing plays available
    rush_budget = max(0, int(round(plays - dropbacks)))

    # --- QB designed rushes (NOT scrambles; scrambles are already in att vs dropbacks) ---
    style = QB_STYLE_CANON.get(canonical_name(qb_params["name"]), {})
    qb_floor = int(round(style.get("rush_att_floor", 0)))
    qb_bias  = style.get("qb_rush_boost", 1.0)
    _, _, to_add, _ = get_penalty_factors(team)
    prs = get_press_factor(def_team)

    qb_design_cap = int(
        max(0, min(0.25 * rush_budget,
                   qb_floor * qb_bias + (1 if prs > 1.05 else 0) + (1 if to_add > 0.06 else 0)))
    )
    qb_rushes = min(qb_design_cap, rush_budget)

    # --- WR gadget runs ---
    wr_frac = _clamp01(team_wr_rush_fraction(team))
    wr_rushes = int(round(rush_budget * wr_frac))
    wr_rushes = min(wr_rushes, rush_budget - qb_rushes)

    # --- RB carries are what's left ---
    rb_carries_total = max(0, rush_budget - qb_rushes - wr_rushes)

    return {
        "plays": plays,
        "attempts": att,
        "sacks": sacks,
        "dropbacks": dropbacks,
        "rush_budget": rush_budget,
        "qb_rushes": qb_rushes,
        "wr_rushes": wr_rushes,
        "rb_carries_total": rb_carries_total,
    }

    # Build team buckets
    rush_info = derive_rush_budget(off_team, qb_params, def_team)

    # RB-only allocation from the RB bucket
    rb_alloc = allocate_rb_carries(off_team, qb_params["name"], def_team,
                               rush_info["rb_carries_total"], "neutral")
    # Distribute QB + RB (+ optional WR jets) to individual names
    rush_by_player = assign_rush_attempts_to_players(off_team, qb_params["name"], roster, rush_info, rb_alloc)

    # When printing each player line:
    p_name_can = canonical_name(p_name)
    rush_att = rush_by_player.get(p_name_can, 0)  # stays 0 for non-rushers


# Who is allowed to rush by default (besides QB)?
ALLOWED_WR_RUSH = {
    # team -> list of WRs who *occasionally* get jets; keep tiny and intentional
    "Los Angeles Rams": ["Puka Nacua", "Tutu Atwell", "Cooper Kupp"],
    "San Francisco 49ers": ["Deebo Samuel"],
    "Miami Dolphins": ["Tyreek Hill", "Jaylen Waddle"],
    # Add more teams/players only if they *actually* get designed carries
}

# Per-position caps (typical NFL reality)
MAX_WR_RUSH_PER_GAME = 2
MAX_TE_RUSH_PER_GAME = 0
MAX_NON_RB_RUSH_PER_GAME = 0  # catch-all safety

def assign_rush_attempts_to_players(
    team: str,
    qb_name: str,
    roster: list[dict],
    rush_budget_info: dict,
    rb_alloc: dict[str, int]
) -> dict[str, int]:
    """
    Returns a dict player_name -> rush attempts for this game.
    Only QB and RBs get attempts by default.
    WR jets are opt-in and strictly capped.
    """
    qb_can = canonical_name(qb_name)
    # roster index for quick lookups
    roster_idx = {canonical_name(e["athlete"]["displayName"]): e for e in roster}

    out = defaultdict(int)

    # 1) QB designed rushes
    qb_runs = int(rush_budget_info.get("qb_rushes", 0))
    if qb_can in roster_idx:
        out[qb_can] = qb_runs

    # 2) RB carries (already integer allocated)
    for rb_name, carries in (rb_alloc or {}).items():
        rb_can = canonical_name(rb_name)
        out[rb_can] += int(carries)

    # 3) WR gadget runs (strict opt-in + tiny volume)
    wr_rushes_total = int(rush_budget_info.get("wr_rushes", 0))
    if wr_rushes_total > 0:
        allowed = [canonical_name(n) for n in ALLOWED_WR_RUSH.get(team, [])]
        # if no explicit list, give *zero* to avoid garbage distribution
        if allowed:
            # simple round-robin up to caps
            i = 0
            given = 0
            caps = {n: MAX_WR_RUSH_PER_GAME for n in allowed}
            while given < wr_rushes_total and any(caps[n] > 0 for n in allowed):
                n = allowed[i % len(allowed)]
                if caps[n] > 0 and n in roster_idx:
                    out[n] += 1
                    caps[n] -= 1
                    given += 1
                i += 1
        # else: silently drop WR gadget runs (can also push them to RB1 if you prefer)

    # 4) Zero out any non-RB/TE accidental rushes (ultra-safety)
    for name in list(out.keys()):
        if name == qb_can:
            continue
        entry = roster_idx.get(name, {})
        pos = get_position_abbr(entry.get("athlete", {}))
        if pos == "RB":
            continue
        if pos == "WR":
            # enforce WR per-player cap (already capped above, but belt & suspenders)
            if out[name] > MAX_WR_RUSH_PER_GAME:
                out[name] = MAX_WR_RUSH_PER_GAME
        elif pos == "TE":
            out[name] = min(out[name], MAX_TE_RUSH_PER_GAME)
        else:
            out[name] = min(out[name], MAX_NON_RB_RUSH_PER_GAME)

    # 5) Integrity: keep within team rush budget
    # (qb + rb + wr must equal rush_budget)
    target = int(rush_budget_info.get("rb_carries_total", 0)) + int(rush_budget_info.get("qb_rushes", 0)) + int(rush_budget_info.get("wr_rushes", 0))
    current = sum(out.values())
    if current != target:
        # Reconcile by nudging RB1 up/down
        depth = TEAM_RB_DEPTH_2025_2026.get(team, [])
        rb1 = canonical_name(depth[0]) if depth else None
        delta = target - current
        if rb1:
            out[rb1] = max(0, out.get(rb1, 0) + delta)
        # After this, sums match target exactly
    return {name: int(v) for name, v in out.items() if v > 0}

    # --- Missing helpers used by derive_rush_budget() ---
def estimate_play_budget(team: str) -> int:
    """Very light fallback. Replace with your real pace model."""
    pace = OFF_TEND.get(team, {}).get("pace", 1.0)
    # 60 plays as base; clamp 50..75
    return max(50, min(75, int(round(60 * pace))))

def expected_dropbacks(qb_params: dict, def_team: str) -> tuple[int, int, int]:
    """
    Return (attempts, sacks, dropbacks). This is a placeholder that keeps
    accounting consistent with your sack-rate model.
    """
    att_mu = int(round(qb_params.get("att_mu", 32)))
    # crude sack estimate from defense pressure + OL baseline
    qb_name = qb_params.get("name", "")
    off_team = qb_params.get("team", "")
    srate = sack_rate_per_dropback(off_team or "DEFAULT", def_team, qb_name or "")
    # dropbacks = attempts / (1 - sack_rate)  => sacks = dropbacks - attempts
    dropbacks = int(round(att_mu / max(0.50, 1.0 - srate)))
    sacks = max(0, dropbacks - att_mu)
    return att_mu, sacks, dropbacks


def apply_player_variance(base_value: float, player_name: str, position: str, matchup_factor: float = 1.0) -> float:
    """Apply enhanced variance for ceiling/floor games"""

    # High variance players who can have breakout games
    high_variance_players = {
        "David Montgomery": 1.4,  # Can have surprise games when leading
        "Mark Andrews": 1.3,     # Boom/bust TE
        "Jahmyr Gibbs": 1.2,
        "Derrick Henry": 1.1
    }

    variance_mult = high_variance_players.get(player_name, 1.0)

    # Position-based variance
    if position == "RB":
        base_variance = base_value * 0.35 * variance_mult
    elif position == "TE":
        base_variance = base_value * 0.40 * variance_mult
    else:
        base_variance = base_value * 0.30 * variance_mult

    # Favorable matchup increases ceiling potential
    if matchup_factor > 1.1:
        base_variance *= 1.2

    return max(0, np.random.normal(base_value, base_variance))



# ---------------------------------------------------
# QB Styles and Attributes
# ---------------------------------------------------
QB_STYLE = {
    # Mobile QBs with scramble modeling
    "Josh Allen": {
        "att_mu": 38, "comp_p": 0.64, "ypa_mu_eff": 7.9, "rush_att_floor": 6,
        "qb_rush_boost": 1.18, "rz_mult": 1.7, "int_lambda": 0.35,
        "scramble_share": 0.40, "scramble_shape": 4.6, "scramble_scale": 2.1,
        "designed_shape": 4.2, "designed_scale": 1.4, "scramble_extra_press_mult": 0.22,
        "td_efficiency": 1.15, "late_game_clutch": 1.25,
    },
    "Lamar Jackson": {
        "att_mu": 30,"comp_p": 0.68, "ypa_mu_eff": 9.2, "rush_att_floor": 8,
        "qb_rush_boost": 1.22, "rz_mult": 1.5, "int_lambda": 0.25,
        "scramble_share": 0.35, "scramble_shape": 4.4, "scramble_scale": 2.0,
        "designed_shape": 4.5, "designed_scale": 1.5, "scramble_extra_press_mult": 0.18,
        "td_efficiency": 1.20, "late_game_clutch": 1.10,
    },
    "J.J. McCarthy": {
        "att_mu": 28, "comp_p": 0.68, "ypa_mu_eff": 7.4, "rush_att_floor": 3,
        "qb_rush_boost": 1.05, "rz_mult": 1.2, "int_lambda": 0.6,
        "scramble_share": 0.40, "scramble_shape": 4.3, "scramble_scale": 1.9,
        "designed_shape": 4.0, "designed_scale": 1.3, "scramble_extra_press_mult": 0.15,
    },
    "Jalen Hurts": {
        "att_mu": 32, "comp_p": 0.69, "ypa_mu_eff": 7.5, "rush_att_floor": 5,
        "qb_rush_boost": 1.10, "rz_mult": 1.25, "int_lambda": 0.5,
        "scramble_share": 0.33, "scramble_shape": 4.3, "scramble_scale": 1.6,
        "designed_shape": 4.4, "designed_scale": 1.35, "scramble_extra_press_mult": 0.12,
    },
    "Kyler Murray": {
        "att_mu": 32, "comp_p": 0.67, "ypa_mu_eff": 7.6, "rush_att_floor": 5,
        "qb_rush_boost": 1.12, "rz_mult": 1.10, "int_lambda": 0.45,
        "scramble_share": 0.42, "scramble_shape": 4.5, "scramble_scale": 1.8,
        "designed_shape": 4.0, "designed_scale": 1.3, "scramble_extra_press_mult": 0.15,
    },
    "Justin Fields": {
        "att_mu": 30, "comp_p": 0.64, "ypa_mu_eff": 7.5, "rush_att_floor": 7,
        "qb_rush_boost": 1.18, "rz_mult": 1.4, "int_lambda": 0.65,
        "scramble_share": 0.45, "scramble_shape": 4.5, "scramble_scale": 2.0,
        "designed_shape": 4.2, "designed_scale": 1.4, "scramble_extra_press_mult": 0.20,
    },
    "Michael Penix Jr.": {
        "att_mu": 34, "comp_p": 0.64, "ypa_mu_eff": 7.1, "int_lambda": 0.68, "sack_lambda_boost": 1.35,
        "development_factor": 1.18,
    },
    "Marcus Mariota": {
        "att_mu": 30, "comp_p": 0.66, "ypa_mu_eff": 6.9, "rush_att_floor": 4, "qb_rush_boost": 1.08,
        "rz_mult": 1.0, "int_lambda": 0.6, "sack_lambda_boost": 1.1,
    },
    # Pocket passers / others
    "Tua Tagovailoa": {
        "att_mu": 32, "comp_p": 0.69, "ypa_mu_eff": 7.1, "ypa_shift": -0.10,
        "ypa_cap": 6.8, "trunc_sigma": 0.20, "trailing_bonus": 0.03, "rz_mult": 1.1,
        "elite_cap": 6.3, "int_lambda": 0.5,
    },
    "Jake Browning": {
        "att_mu": 35, "comp_p": 0.68, "ypa_mu_eff": 7.5, "int_lambda": 0.55, "sack_lambda_boost": 1.25,
    },
    "Joe Flacco": {
        "att_mu": 33, "comp_p": 0.65, "ypa_mu_eff": 7.0, "int_lambda": 0.8, "sack_lambda_boost": 1.3,
        "age_decline": 0.90,
    },
    "Patrick Mahomes": {
        "att_mu": 36, "comp_p": 0.67, "ypa_mu_eff": 7.8, "rush_att_floor": 3, "qb_rush_boost": 1.05,
        "rz_mult": 1.3, "int_lambda": 0.45,
        "scramble_share": 0.28, "scramble_extra_press_mult": 0.12, "td_efficiency": 1.25, "late_game_clutch": 1.35,
    },
    "Caleb Williams": {
        "att_mu": 34, "comp_p": 0.63, "ypa_mu_eff": 7.0, "int_lambda": 0.65, "sack_lambda_boost": 1.3,
        "development_factor": 1.15,
    },
    "Bo Nix": {
        "att_mu": 34, "comp_p": 0.69, "ypa_mu_eff": 7.4, "int_lambda": 0.7, "sack_lambda_boost": 1.15, "rz_mult": 1.40,
        "development_factor": 1.25,
    },
    "Drake Maye": {
        "att_mu": 30, "comp_p": 0.62, "ypa_mu_eff": 6.9, "int_lambda": 0.75, "sack_lambda_boost": 1.4,
        "development_factor": 1.10,
    },
    "Aidan O'Connell": {
        "att_mu": 28, "comp_p": 0.64, "ypa_mu_eff": 6.8, "int_lambda": 0.7, "sack_lambda_boost": 1.3,
    },
    "C.J. Stroud": {
        "att_mu": 30, "comp_p": 0.66, "ypa_mu_eff": 7.0, "int_lambda": 0.6, "sack_lambda_boost": 1.2,
        "injury_adjustment": 0.95,
    },
    "Cam Ward": {
        "att_mu": 30, "comp_p": 0.65, "ypa_mu_eff": 7.2, "int_lambda": 0.68, "sack_lambda_boost": 1.25,
        "rookie_development": 1.10,
    },
    "Daniel Jones": {
        "att_mu": 34, "comp_p": 0.68, "ypa_mu_eff": 8.3, "rz_mult": 0.95, "int_lambda": 0.5,
        "rush_att_floor": 5, "qb_rush_boost": 1.08, "sack_lambda_boost": 1.05, "rush_td_boost": 1.15, "sneak_p": 0.25,
    },
    # Veterans with 2025 adjustments
    "Aaron Rodgers": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Russell Wilson": {
        "att_mu": 33, "comp_p": 0.66, "ypa_mu_eff": 7.1, "int_lambda": 0.48,
        "rush_att_floor": 4, "qb_rush_boost": 1.08, "age_decline": 0.93
    },
    "Dak Prescott": {
        "att_mu": 34, "comp_p": 0.67, "ypa_mu_eff": 7.3, "int_lambda": 0.55,
        "rush_att_floor": 4, "qb_rush_boost": 1.07, "sack_lambda_boost": 1.1,
    },
    "Jared Goff": {
        "att_mu": 30, "comp_p": 0.66, "ypa_mu_eff": 7.0, "int_lambda": 0.6, "sack_lambda_boost": 1.2,
    },
    "Baker Mayfield": {
        "att_mu": 28, "comp_p": 0.64, "ypa_mu_eff": 6.8, "int_lambda": 0.75, "sack_lambda_boost": 1.3,
        "scramble_share": 0.18, "scramble_extra_press_mult": 0.12, "late_game_clutch": 1.08,
    },
    "Jordan Love": {
        "att_mu": 30, "comp_p": 0.65, "ypa_mu_eff": 7.1, "int_lambda": 0.7, "sack_lambda_boost": 1.2,
    },
    "Sam Darnold": {
        "att_mu": 31, "comp_p": 0.65, "ypa_mu_eff": 7.4, "rz_mult": 1.1, "int_lambda": 0.7, "sack_lambda_boost": 1.3,
    },
    # Injured Players
    "Joe Burrow": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Anthony Richardson": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Jayden Daniels": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Will Levis": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Kirk Cousins": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
    "Deshaun Watson": {
        "att_mu": 0.0, "comp_p": 0.0, "ypa_mu_eff": 0.0, "int_lambda": 0.0, "sack_lambda_boost": 0.0,
        "injury_status": "IR", "out_for_season": True,
    },
}

# === Name canonicalization (place right after QB_STYLE and RZ_WEIGHTS) ===
def canonical_name(s: str) -> str:
    s = s.replace("’", "'")
    s = s.replace(".", "")              # ← remove ALL periods in names
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\b([A-Z])\s+([A-Z])\b", r"\1\2", s)  # "A J" -> "AJ"
    return s

QB_STYLE_CANON = {canonical_name(k): v for k, v in QB_STYLE.items()}
RZ_WEIGHTS_CANON = {
    team: {canonical_name(p): w for p, w in dd.items()}
    for team, dd in RZ_WEIGHTS.items()
}


MOBILE_QB_SET = {
    "Josh Allen", "Lamar Jackson", "Jalen Hurts", "Justin Fields", "Kyler Murray",
    "Anthony Richardson", "Jayden Daniels"
}

PLAYER_BASELINES = {}
RB_SYNERGY_YPC_BOOST = 1.08  # RB YPC lift with mobile QB

# ---------------------------------------------------
# Focus players — all 32 teams
# ---------------------------------------------------
FOCUS_PLAYERS = {
    "Buffalo Bills": ["Josh Allen", "James Cook", "Khalil Shakir", "Keon Coleman", "Dalton Kincaid", "Joshua Palmer", "Ray Davis"],
    "Miami Dolphins": ["Tua Tagovailoa", "De'Von Achane", "Tyreek Hill", "Jaylen Waddle", "Malik Washington"],
    "New York Jets": ["Justin Fields", "Breece Hall", "Garrett Wilson", "Mike Williams", "Tyler Conklin", "Malachi Corley"],
    "New England Patriots": ["Drake Maye", "Rhamondre Stevenson", "Stefon Diggs", "Kendrick Bourne", "Hunter Henry", "Kayshon Boutte", "TreVeyon Henderson"],
    "Baltimore Ravens": ["Lamar Jackson", "Derrick Henry", "Zay Flowers", "Mark Andrews", "Rashod Bateman", "DeAndre Hopkins", "Justice Hill"],
    "Cincinnati Bengals": ["Jake Browning", "Chase Brown", "Ja'Marr Chase", "Tee Higgins", "Mike Gesicki", "Andrei Iosivas"],
    "Cleveland Browns": ["Joe Flacco", "Quinshon Judkins", "Amari Cooper", "Jerry Jeudy", "David Njoku", "Elijah Moore", "Jerome Ford"],
    "Pittsburgh Steelers": ["Aaron Rodgers", "Jaylen Warren", "DK Metcalf", "Pat Freiermuth", "Roman Wilson", "Calvin Austin III", "Kenneth Gainwell"],
    "Houston Texans": ["C.J. Stroud", "Nick Chubb", "Nico Collins", "Christian Kirk", "Tank Dell", "Dalton Schultz", "Noah Brown"],
    "Indianapolis Colts": ["Daniel Jones", "Jonathan Taylor", "Michael Pittman Jr", "Josh Downs", "Adonai Mitchell", "Jelani Woods", "Alec Pierce"],
    "Jacksonville Jaguars": ["Trevor Lawrence", "Travis Etienne", "Brian Thomas Jr", "Evan Engram", "Dyami Brown", "Travis Hunter"],
    "Tennessee Titans": ["Cam Ward", "Tony Pollard", "Calvin Ridley", "Tyjae Spears", "Chigoziem Okonkwo", "Tyler Lockett", "Elic Ayomanor"],
    "Denver Broncos": ["Bo Nix", "J.K. Dobbins", "Courtland Sutton", "Marvin Mims Jr", "Troy Franklin", "Greg Dulcich", "Javonte Williams"],
    "Kansas City Chiefs": ["Patrick Mahomes", "Isiah Pacheco", "Travis Kelce", "Marquise Brown", "JuJu Smith-Schuster", "Xavier Worthy", "Noah Gray"],
    "Las Vegas Raiders": ["Geno Smith", "Ashton Jeanty", "Zamir White", "Jakobi Meyers", "Tre Tucker", "Brock Bowers"],
    "Los Angeles Chargers": ["Justin Herbert", "Omarion Hampton", "Keenan Allen", "Ladd McConkey", "Quentin Johnston", "Will Dissly", "Gus Edwards"],
    "Dallas Cowboys": ["Dak Prescott", "Javonte Williams", "George Pickens", "Brandin Cooks", "Jake Ferguson", "Jalen Tolbert", "Ezekiel Elliott"],
    "New York Giants": [
        "Russell Wilson",
        "Cam Skattebo",
        "Theo Johnson",
        "Malik Nabers",
        "Wan'Dale Robinson",
        "Darius Slayton",
    ],
    "Philadelphia Eagles": ["Jalen Hurts", "Saquon Barkley", "A.J. Brown", "DeVonta Smith", "Dallas Goedert", "Jahan Dotson", "Will Shipley"],
    "Washington Commanders": ["Marcus Mariota", "Deebo Samuel", "Terry McLaurin", "Zach Ertz", "Luke McCaffrey", "Jacory Croskey-Merritt", "Chris Rodriguez Jr."],
    "Chicago Bears": ["Caleb Williams", "D'Andre Swift", "DJ Moore", "Rome Odunze", "Cole Kmet", "Olamide Zaccheaus"],
    "Detroit Lions": ["Jared Goff", "Jahmyr Gibbs", "Amon-Ra St. Brown", "Sam LaPorta", "Jameson Williams", "David Montgomery"],
    "Green Bay Packers": ["Jordan Love", "Josh Jacobs", "Romeo Doubs", "Dontayvion Wicks", "Luke Musgrave", "Matthew Golden"],
    "Minnesota Vikings": ["J.J. McCarthy", "Jordan Mason", "Justin Jefferson", "Jordan Addison", "T.J. Hockenson", "Zavier Scott"],
    "Atlanta Falcons": ["Michael Penix Jr", "Bijan Robinson", "Drake London", "Kyle Pitts", "Darnell Mooney"],
    "Carolina Panthers": ["Bryce Young", "Chuba Hubbard", "Hunter Renfrow", "Xavier Legette", "Ja'Tavion Sanders", "Tetairoa McMillan"],
    "New Orleans Saints": ["Spencer Rattler", "Alvin Kamara", "Chris Olave", "Rashid Shaheed", "Juwan Johnson", "Kendre Miller"],
    "Tampa Bay Buccaneers": ["Baker Mayfield", "Bucky Irving", "Chris Godwin", "Rachaad White", "Emeka Egbuka", "Sterling Shepard"],
    "Arizona Cardinals": ["Kyler Murray", "Trey Benson", "Marvin Harrison Jr", "Trey McBride", "Michael Wilson", "Greg Dortch"],
    "Los Angeles Rams": ["Matthew Stafford", "Kyren Williams", "Puka Nacua", "Cooper Kupp", "Davante Adams", "Tyler Higbee"],
    "San Francisco 49ers": ["Brock Purdy", "Christian McCaffrey", "Jauan Jennings", "Ricky Pearsall", "George Kittle"],
    "Seattle Seahawks": ["Sam Darnold", "Kenneth Walker III", "Zach Charbonnet", "Jaxon Smith-Njigba", "Cooper Kupp", "Tyler Boyd", "Noah Fant"],
}

# ---------------------------------------------------
# Offensive Line Protection Grades
# ---------------------------------------------------
O_LINE_PROTECTION = {
    "Buffalo Bills": {"grade": 0.78, "sack_rate": 0.055, "pressure_allowed": 0.26},
    "Miami Dolphins": {"grade": 0.69, "sack_rate": 0.080, "pressure_allowed": 0.35},  # Downgraded due to poor recent performance and injuries
    "New York Jets": {"grade": 0.69, "sack_rate": 0.072, "pressure_allowed": 0.32},
    "New England Patriots": {"grade": 0.62, "sack_rate": 0.082, "pressure_allowed": 0.36},
    "Baltimore Ravens": {"grade": 0.73, "sack_rate": 0.065, "pressure_allowed": 0.31},  # Downgraded due to sacks allowed vs. Lions
    "Cleveland Browns": {"grade": 0.67, "sack_rate": 0.075, "pressure_allowed": 0.33},
    "Cincinnati Bengals": {"grade": 0.73, "sack_rate": 0.061, "pressure_allowed": 0.29},
    "Pittsburgh Steelers": {"grade": 0.70, "sack_rate": 0.065, "pressure_allowed": 0.31},
    "Houston Texans": {"grade": 0.62, "sack_rate": 0.085, "pressure_allowed": 0.37},  # Downgraded due to injuries and poor play
    "Jacksonville Jaguars": {"grade": 0.64, "sack_rate": 0.078, "pressure_allowed": 0.35},
    "Indianapolis Colts": {"grade": 0.68, "sack_rate": 0.073, "pressure_allowed": 0.33},
    "Tennessee Titans": {"grade": 0.63, "sack_rate": 0.080, "pressure_allowed": 0.36},
    "Kansas City Chiefs": {"grade": 0.74, "sack_rate": 0.060, "pressure_allowed": 0.29},
    "Los Angeles Chargers": {"grade": 0.76, "sack_rate": 0.070, "pressure_allowed": 0.28},  # Upgraded due to early season success, but with a higher sack rate
    "Denver Broncos": {"grade": 0.81, "sack_rate": 0.052, "pressure_allowed": 0.24},  # Upgraded based on multiple high ratings
    "Las Vegas Raiders": {"grade": 0.65, "sack_rate": 0.077, "pressure_allowed": 0.34},
    "Philadelphia Eagles": {"grade": 0.83, "sack_rate": 0.051, "pressure_allowed": 0.24},  # Retains top spot with a slight upgrade
    "Dallas Cowboys": {"grade": 0.71, "sack_rate": 0.067, "pressure_allowed": 0.31},
    "New York Giants": {"grade": 0.60, "sack_rate": 0.085, "pressure_allowed": 0.38},
    "Washington Commanders": {"grade": 0.69, "sack_rate": 0.071, "pressure_allowed": 0.32},
    "Detroit Lions": {"grade": 0.82, "sack_rate": 0.053, "pressure_allowed": 0.25},
    "Green Bay Packers": {"grade": 0.73, "sack_rate": 0.062, "pressure_allowed": 0.29},
    "Chicago Bears": {"grade": 0.61, "sack_rate": 0.083, "pressure_allowed": 0.37},
    "Minnesota Vikings": {"grade": 0.70, "sack_rate": 0.066, "pressure_allowed": 0.31},
    "Atlanta Falcons": {"grade": 0.68, "sack_rate": 0.072, "pressure_allowed": 0.32},
    "Carolina Panthers": {"grade": 0.65, "sack_rate": 0.075, "pressure_allowed": 0.33},
    "New Orleans Saints": {"grade": 0.72, "sack_rate": 0.064, "pressure_allowed": 0.30},
    "Tampa Bay Buccaneers": {"grade": 0.74, "sack_rate": 0.059, "pressure_allowed": 0.28},
    "San Francisco 49ers": {"grade": 0.76, "sack_rate": 0.057, "pressure_allowed": 0.27},
    "Seattle Seahawks": {"grade": 0.67, "sack_rate": 0.074, "pressure_allowed": 0.33},
    "Los Angeles Rams": {"grade": 0.69, "sack_rate": 0.070, "pressure_allowed": 0.32},
    "Arizona Cardinals": {"grade": 0.72, "sack_rate": 0.063, "pressure_allowed": 0.29},
}

def sack_rate_per_dropback(off_team: str, def_team: str, qb_name: str) -> float:
    """Enhanced sack rate calculation with pass rush win rates"""

    ol = O_LINE_PROTECTION.get(off_team, {"grade": 0.66, "sack_rate": 0.075})

    # Enhanced pressure factors for specific matchups
    enhanced_pressure = {
        "Detroit Lions": 1.20,  # Strong pass rush
        "Baltimore Ravens": 0.85,  # Weak pass rush based on analysis
        "Philadelphia Eagles": 1.18,
        "Cleveland Browns": 1.15
    }

    prs = enhanced_pressure.get(def_team, get_press_factor(def_team))
    base = float(ol.get("sack_rate", 0.075))

    style = QB_STYLE_CANON.get(canonical_name(qb_name), {})
    hold = 1.0 + 0.05 * (style.get("ypa_mu_eff", 7.0) < 7.0) + 0.05 * (style.get("att_mu", 32) > 35)

    # Mobile QB escape ability
    mobile_qbs = {"Lamar Jackson", "Josh Allen", "Jalen Hurts", "Anthony Richardson",
                  "Justin Fields", "Jayden Daniels", "Kyler Murray"}
    esc = 0.85 if qb_name in mobile_qbs else 1.00

    grade = float(ol.get("grade", 0.66))
    undergrade_mult = 1.05 + max(0.0, 0.78 - grade) * 1.10

    adj = base * prs * hold * esc * undergrade_mult
    return float(max(0.02, min(0.16, adj)))

OFF_TEND = {
    "Arizona Cardinals": {"pace": 1.01, "pass_rate": 1.02, "red_zone_aggression": 1.0},
    "Atlanta Falcons": {"pace": 1.05, "pass_rate": 1.08, "red_zone_aggression": 1.1}, # More pass-heavy than projected due to QB struggles
    "Baltimore Ravens": {"pace": 1.08, "pass_rate": 1.06, "red_zone_aggression": 1.40},
    "Buffalo Bills": {"pace": 1.08, "pass_rate": 1.10, "red_zone_aggression": 1.35},
    "Carolina Panthers": {"pace": 0.98, "pass_rate": 0.92, "red_zone_aggression": 0.9},
    "Chicago Bears": {"pace": 0.98, "pass_rate": 1.00, "red_zone_aggression": 0.95},
    "Cincinnati Bengals": {"pace": 1.00, "pass_rate": 1.12, "red_zone_aggression": 1.15}, # More pass-heavy due to Browning, despite struggles
    "Cleveland Browns": {"pace": 1.04, "pass_rate": 1.10, "red_zone_aggression": 1.1}, # More pass-heavy with Flacco
    "Dallas Cowboys": {"pace": 1.08, "pass_rate": 1.08, "red_zone_aggression": 1.15}, # Faster pace and more pass-heavy
    "Denver Broncos": {"pace": 1.02, "pass_rate": 0.92, "red_zone_aggression": 1.1},
    "Detroit Lions": {"pace": 0.98, "pass_rate": 1.00, "red_zone_aggression": 1.20}, # Slower pace than expected
    "Green Bay Packers": {"pace": 1.01, "pass_rate": 1.02, "red_zone_aggression": 1.05},
    "Houston Texans": {"pace": 0.99, "pass_rate": 0.98, "red_zone_aggression": 0.95}, # Slower pace, more balanced offense
    "Indianapolis Colts": {"pace": 1.00, "pass_rate": 1.02, "red_zone_aggression": 1.05},
    "Jacksonville Jaguars": {"pace": 1.03, "pass_rate": 1.01, "red_zone_aggression": 1.05}, # Slightly faster pace
    "Kansas City Chiefs": {"pace": 1.03, "pass_rate": 0.98, "red_zone_aggression": 1.05}, # Lower pass rate than original projection
    "Las Vegas Raiders": {"pace": 0.97, "pass_rate": 0.96, "red_zone_aggression": 0.9},
    "Los Angeles Chargers": {"pace": 1.05, "pass_rate": 1.03, "red_zone_aggression": 1.05}, # Faster pace
    "Los Angeles Rams": {"pace": 1.03, "pass_rate": 1.04, "red_zone_aggression": 1.1},
    "Miami Dolphins": {"pace": 1.00, "pass_rate": 1.00, "red_zone_aggression": 1.10},
    "Minnesota Vikings": {"pace": 1.05, "pass_rate": 1.08, "red_zone_aggression": 1.2},
    "New England Patriots": {"pace": 0.96, "pass_rate": 0.98, "red_zone_aggression": 0.9},
    "New Orleans Saints": {"pace": 1.08, "pass_rate": 1.01, "red_zone_aggression": 1.0}, # Faster pace
    "New York Giants": {"pace": 1.05, "pass_rate": 1.08, "red_zone_aggression": 1.20}, # More aggressive with Russell Wilson
    "New York Jets": {"pace": 0.98, "pass_rate": 1.02, "red_zone_aggression": 1.0},
    "Philadelphia Eagles": {"pace": 1.02, "pass_rate": 0.98, "red_zone_aggression": 1.0},
    "Pittsburgh Steelers": {"pace": 0.98, "pass_rate": 1.05, "red_zone_aggression": 1.05}, # More pass-heavy with Rodgers
    "San Francisco 49ers": {"pace": 1.01, "pass_rate": 0.97, "red_zone_aggression": 1.0},
    "Seattle Seahawks": {"pace": 1.02, "pass_rate": 1.04, "red_zone_aggression": 1.10},
    "Tampa Bay Buccaneers": {"pace": 1.04, "pass_rate": 1.07, "red_zone_aggression": 1.15},
    "Tennessee Titans": {"pace": 0.96, "pass_rate": 0.94, "red_zone_aggression": 0.85},
    "Washington Commanders": {"pace": 1.00, "pass_rate": 0.90, "red_zone_aggression": 0.8}, # Significantly more run-heavy
}

# WR jet/endo-around usage
TEAM_WR_RUSH_RATE = {
    "Los Angeles Rams": 0.06,
    "San Francisco 49ers": 0.03,
    "Miami Dolphins": 0.03,
}

WR_RUSH_WEIGHTS = {
    "Los Angeles Rams": {"Puka Nacua": 0.55, "Tutu Atwell": 0.25, "Cooper Kupp": 0.10, "Other": 0.10},
}


# ---------------------------------------------------
# QB Overrides for weekly starting QBs
# ---------------------------------------------------
QB_OVERRIDES = {
    "Arizona Cardinals": "Kyler Murray",
    "Atlanta Falcons": "Michael Penix Jr",
    "Baltimore Ravens": "Lamar Jackson",
    "Buffalo Bills": "Josh Allen",
    "Carolina Panthers": "Bryce Young",
    "Chicago Bears": "Caleb Williams",
    "Cincinnati Bengals": "Jake Browning",
    "Cleveland Browns": "Joe Flacco",
    "Dallas Cowboys": "Dak Prescott",
    "Denver Broncos": "Bo Nix",
    "Detroit Lions": "Jared Goff",
    "Green Bay Packers": "Jordan Love",
    "Houston Texans": "CJ Stroud",
    "Indianapolis Colts": "Daniel Jones",
    "Jacksonville Jaguars": "Trevor Lawrence",
    "Kansas City Chiefs": "Patrick Mahomes",
    "Las Vegas Raiders": "Geno Smith",
    "Los Angeles Chargers": "Justin Herbert",
    "Los Angeles Rams": "Matthew Stafford",
    "Miami Dolphins": "Tua Tagovailoa",
    "Minnesota Vikings": "J.J. McCarthy",
    "New England Patriots": "Drake Maye",
    "New Orleans Saints": "Spencer Rattler",
    "New York Giants": "Jaxson Dart",
    "New York Jets": "Justin Fields",
    "Philadelphia Eagles": "Jalen Hurts",
    "Pittsburgh Steelers": "Aaron Rodgers",
    "San Francisco 49ers": "Mac Jones",
    "Seattle Seahawks": "Sam Darnold",
    "Tampa Bay Buccaneers": "Baker Mayfield",
    "Tennessee Titans": "Cam Ward",
    "Washington Commanders": None
}

# ---------------------------------------------------
# Defensive adjustments (DVOA/EPA-informed baseline)
# ---------------------------------------------------
# DEF_ADJ = {
#     # Elite pass defenses (major downgrades for opposing passing)
#     "Denver Broncos": {"short":0.83,"deep":0.94,"te":0.88,"rush":0.94,"xpl":0.80,"press":1.18},
#     "Cleveland Browns": {"short":0.88,"deep":0.85,"te":0.90,"rush":0.88,"xpl":0.82,"press":1.12},
#     "Pittsburgh Steelers": {"pass": 0.86, "rush": 0.92, "pressure": 1.20, "coverage": 0.82},
#     "New Orleans Saints": {"short":0.91,"deep":0.89,"te":0.92,"rush":1.05,"xpl":0.88,"press":1.08},
#     "Indianapolis Colts": {"short":0.90,"deep":0.88,"te":0.91,"rush":0.95,"xpl":0.85,"press":1.10},
#     "Chicago Bears": {"short":0.89,"deep":0.87,"te":0.90,"rush":0.99,"xpl":0.86,"press":1.09},

#     # Vulnerable pass defenses (major upgrades for opposing passing)
#     "Baltimore Ravens": {"short":1.15,"deep":1.18,"te":1.12,"rush":1.08,"xpl":1.15,"press":0.88},
#     "New England Patriots": {"short":1.12,"deep":1.15,"te":1.10,"rush":0.95,"xpl":1.12,"press":0.90},
#     "Kansas City Chiefs": {"short":1.08,"deep":1.12,"te":1.06,"rush":1.02,"xpl":1.08,"press":0.92},
#     "Tampa Bay Buccaneers": {"short":1.06,"deep":1.10,"te":1.04,"rush":0.96,"xpl":1.06,"press":0.94},

#     # Elite rush defenses
#     "Green Bay Packers": {"short":0.98,"deep":0.96,"te":0.97,"rush":0.85,"xpl":0.90,"press":1.08},
#     "Cincinnati Bengals": {"short":1.00,"deep":0.98,"te":0.99,"rush":0.87,"xpl":0.92,"press":1.05},
#     "New York Jets": {"short":0.94,"deep":0.92,"te":0.94,"rush":0.89,"xpl":0.88,"press":1.10},

#     # Vulnerable rush defenses
#     "Buffalo Bills": {"short":0.95,"deep":0.97,"te":0.96,"rush":1.20,"xpl":0.94,"press":1.02},
#     "New York Giants": {"short":1.02,"deep":1.04,"te":1.03,"rush":1.18,"xpl":1.04,"press":0.96},
#     "Carolina Panthers": {"short":1.04,"deep":1.06,"te":1.05,"rush":1.15,"xpl":1.06,"press":0.94},
# }

# ELITE_PASS_DEF = {
#     "Buffalo Bills", "Baltimore Ravens", "Cleveland Browns", "San Francisco 49ers",
#     "Dallas Cowboys", "New York Jets", "Detroit Lions", "Kansas City Chiefs", "Green Bay Packers"
# }

# # Enhanced pressure data (present but currently unused in core loop)
# ENHANCED_PRESSURE_DATA = {
#     "Buffalo Bills": {"pressure_rate": 0.35, "sack_conversion": 0.08, "qb_disruption": 1.3},
#     "Miami Dolphins": {"pressure_rate": 0.26, "sack_conversion": 0.065, "qb_disruption": 0.9},
#     "New York Jets": {"pressure_rate": 0.34, "sack_conversion": 0.085, "qb_disruption": 1.2},
#     "New England Patriots": {"pressure_rate": 0.22, "sack_conversion": 0.055, "qb_disruption": 0.8},
#     "Baltimore Ravens": {"pressure_rate": 0.32, "sack_conversion": 0.082, "qb_disruption": 1.2},
#     "Cleveland Browns": {"pressure_rate": 0.33, "sack_conversion": 0.088, "qb_disruption": 1.3},
#     "Cincinnati Bengals": {"pressure_rate": 0.24, "sack_conversion": 0.06, "qb_disruption": 0.9},
#     "Pittsburgh Steelers": {"pressure_rate": 0.29, "sack_conversion": 0.072, "qb_disruption": 1.0},
#     "Houston Texans": {"pressure_rate": 0.28, "sack_conversion": 0.07, "qb_disruption": 1.0},
#     "Jacksonville Jaguars": {"pressure_rate": 0.25, "sack_conversion": 0.062, "qb_disruption": 0.9},
#     "Indianapolis Colts": {"pressure_rate": 0.23, "sack_conversion": 0.058, "qb_disruption": 0.8},
#     "Tennessee Titans": {"pressure_rate": 0.24, "sack_conversion": 0.061, "qb_disruption": 0.9},
#     "Kansas City Chiefs": {"pressure_rate": 0.31, "sack_conversion": 0.078, "qb_disruption": 1.1},
#     "Los Angeles Chargers": {"pressure_rate": 0.27, "sack_conversion": 0.068, "qb_disruption": 0.95},
#     "Denver Broncos": {"pressure_rate": 0.26, "sack_conversion": 0.065, "qb_disruption": 0.9},
#     "Las Vegas Raiders": {"pressure_rate": 0.25, "sack_conversion": 0.063, "qb_disruption": 0.9},
#     "Philadelphia Eagles": {"pressure_rate": 0.29, "sack_conversion": 0.073, "qb_disruption": 1.0},
#     "Dallas Cowboys": {"pressure_rate": 0.32, "sack_conversion": 0.081, "qb_disruption": 1.2},
#     "New York Giants": {"pressure_rate": 0.23, "sack_conversion": 0.057, "qb_disruption": 0.8},
#     "Washington Commanders": {"pressure_rate": 0.21, "sack_conversion": 0.052, "qb_disruption": 0.75},
#     "Detroit Lions": {"pressure_rate": 0.30, "sack_conversion": 0.076, "qb_disruption": 1.1},
#     "Green Bay Packers": {"pressure_rate": 0.31, "sack_conversion": 0.079, "qb_disruption": 1.15},
#     "Chicago Bears": {"pressure_rate": 0.27, "sack_conversion": 0.068, "qb_disruption": 0.95},
#     "Minnesota Vikings": {"pressure_rate": 0.26, "sack_conversion": 0.066, "qb_disruption": 0.9},
#     "Atlanta Falcons": {"pressure_rate": 0.30, "sack_conversion": 0.075, "qb_disruption": 1.05},
#     "Carolina Panthers": {"pressure_rate": 0.28, "sack_conversion": 0.072, "qb_disruption": 1.0},
#     "New Orleans Saints": {"pressure_rate": 0.28, "sack_conversion": 0.071, "qb_disruption": 1.0},
#     "Tampa Bay Buccaneers": {"pressure_rate": 0.29, "sack_conversion": 0.074, "qb_disruption": 1.0},
#     "San Francisco 49ers": {"pressure_rate": 0.33, "sack_conversion": 0.084, "qb_disruption": 1.25},
#     "Seattle Seahawks": {"pressure_rate": 0.24, "sack_conversion": 0.061, "qb_disruption": 0.85},
#     "Los Angeles Rams": {"pressure_rate": 0.28, "sack_conversion": 0.070, "qb_disruption": 0.98},
#     "Arizona Cardinals": {"pressure_rate": 0.24, "sack_conversion": 0.060, "qb_disruption": 0.85},
# }

# ---------------------------------------------------
# Recent-game / Penalty adjustments (small, data-driven nudges)
# ---------------------------------------------------
# ---------------------------------------------------
# Penalty Baseline (must always exist for sim safety)
# ---------------------------------------------------
TEAM_PENALTY_BASE = {
    # baseline: ~6 penalties, ~45 yards
    # override per team if you want more realism
    "DEFAULT": {"pen_count": 6, "pen_yards": 45},
    "Tennessee Titans": {"pen_count": 11.5, "pen_yards": 96},  # reflects early-season trend
    "Los Angeles Rams": {"pen_count": 4.0,  "pen_yards": 30},
    "New York Giants": {"pen_count": 4.5,  "pen_yards": 32},
    "Cleveland Browns": {"pen_count": 4.8,  "pen_yards": 34},
    "Philadelphia Eagles": {"pen_count": 5.0,  "pen_yards": 36},
    "Dallas Cowboys": {"pen_count": 5.2,  "pen_yards": 38},
    "New England Patriots": {"pen_count": 5.5,  "pen_yards": 40},
    "Detroit Lions": {"pen_count": 5.8,  "pen_yards": 42},
    "Kansas City Chiefs": {"pen_count": 5.9,  "pen_yards": 43},
    "Miami Dolphins": {"pen_count": 6.2,  "pen_yards": 47},
    "Baltimore Ravens": {"pen_count": 6.5,  "pen_yards": 50},
    "Green Bay Packers": {"pen_count": 6.8,  "pen_yards": 52},
    "New York Jets": {"pen_count": 7.0,  "pen_yards": 55},
    "Washington Commanders": {"pen_count": 7.5,  "pen_yards": 58},
    "Atlanta Falcons": {"pen_count": 8.0,  "pen_yards": 60},
    "Tampa Bay Buccaneers": {"pen_count": 8.5,  "pen_yards": 65},
    "Indianapolis Colts": {"pen_count": 9.0,  "pen_yards": 70},
    "Chicago Bears": {"pen_count": 9.5,  "pen_yards": 75},
    "Cincinnati Bengals": {"pen_count": 10.0, "pen_yards": 80},
    "Pittsburgh Steelers": {"pen_count": 10.5, "pen_yards": 85},
    "Jacksonville Jaguars": {"pen_count": 11.0, "pen_yards": 90},
    "Minnesota Vikings": {"pen_count": 4.2,  "pen_yards": 33},
    "Los Angeles Chargers": {"pen_count": 5.4,  "pen_yards": 39},
    "San Francisco 49ers": {"pen_count": 4.7,  "pen_yards": 35},
    "Seattle Seahawks": {"pen_count": 6.0,  "pen_yards": 45},
    "Arizona Cardinals": {"pen_count": 7.2,  "pen_yards": 57},
    "Houston Texans": {"pen_count": 8.3,  "pen_yards": 62},
    "Carolina Panthers": {"pen_count": 9.1,  "pen_yards": 72},
    "Denver Broncos": {"pen_count": 10.2, "pen_yards": 78},
    "New Orleans Saints": {"pen_count": 11.3, "pen_yards": 95},
    "Buffalo Bills": {"pen_count": 5.6,  "pen_yards": 41},
}

def get_penalty_factors(team_name: str):
    p = TEAM_PENALTY_BASE.get(team_name, TEAM_PENALTY_BASE["DEFAULT"])
    pen_count = float(p.get("pen_count", 6))
    pen_yards = float(p.get("pen_yards", 45))
    baseline = 6.0

    extra = max(0.0, pen_count - baseline)
    turnover_add = min(0.45, 0.02 * extra)
    yard_impact = 1.0 + max(0.0, (pen_yards - 40.0) / 200.0)

    return pen_count, pen_yards, turnover_add, yard_impact
# ---------------------------------------------------
# Penalty Baseline (must always exist for sim safety)



# ---------------------------------------------------
# ENHANCED Defensive adjustments (2025 season data-informed)
# Based on current NFL performance through September 2025
# ---------------------------------------------------

DEF_ADJ_2025 = {
    # ELITE PASS DEFENSES
    "Cleveland Browns": {
        "short": 0.85, "deep": 0.82, "te": 0.87, "rush": 0.90,
        "xpl": 0.78, "press": 1.15, "coverage_grade": 0.85
    },
    "Denver Broncos": {
        "short": 0.86, "deep": 0.88, "te": 0.89, "rush": 0.96,
        "xpl": 0.82, "press": 1.12, "coverage_grade": 0.87
    },
    "Green Bay Packers": {
        "short": 0.88, "deep": 0.85, "te": 0.90, "rush": 0.88,
        "xpl": 0.80, "press": 1.18, "coverage_grade": 0.86
    },
    "Philadelphia Eagles": {
        "short": 0.84, "deep": 0.81, "te": 0.86, "rush": 0.85,
        "xpl": 0.77, "press": 1.20, "coverage_grade": 0.83
    },
    "Pittsburgh Steelers": {
        "short": 0.92, "deep": 0.90, "te": 0.93, "rush": 0.94,
        "xpl": 0.88, "press": 1.08, "coverage_grade": 0.91
    },

    # STRONG DEFENSES
    "New York Jets": {
        "short": 0.91, "deep": 0.89, "te": 0.92, "rush": 0.87,
        "xpl": 0.85, "press": 1.12, "coverage_grade": 0.90
    },
    "Indianapolis Colts": {
        "short": 0.89, "deep": 0.87, "te": 0.90, "rush": 0.93,
        "xpl": 0.84, "press": 1.10, "coverage_grade": 0.88
    },
    "Chicago Bears": {
        "short": 0.90, "deep": 0.88, "te": 0.91, "rush": 0.97,
        "xpl": 0.86, "press": 1.09, "coverage_grade": 0.89
    },
    "Houston Texans": {
        "short": 0.93, "deep": 0.91, "te": 0.94, "rush": 0.89,
        "xpl": 0.87, "press": 1.10, "coverage_grade": 0.92
    },
    "Los Angeles Chargers": {
        "short": 0.95, "deep": 0.97, "te": 0.96, "rush": 1.01,
        "xpl": 0.95, "press": 1.03, "coverage_grade": 0.96
    },

    # VULNERABLE PASS DEFENSES
    "New England Patriots": {
        "short": 1.18, "deep": 1.22, "te": 1.15, "rush": 0.98,
        "xpl": 1.20, "press": 0.85, "coverage_grade": 1.18
    },
    "Jacksonville Jaguars": {
        "short": 1.15, "deep": 1.18, "te": 1.12, "rush": 1.15,
        "xpl": 1.16, "press": 0.88, "coverage_grade": 1.15
    },
    "Carolina Panthers": {
        "short": 1.12, "deep": 1.15, "te": 1.10, "rush": 1.18,
        "xpl": 1.12, "press": 0.90, "coverage_grade": 1.12
    },
    "Tennessee Titans": {
        "short": 1.10, "deep": 1.13, "te": 1.08, "rush": 1.12,
        "xpl": 1.10, "press": 0.92, "coverage_grade": 1.10
    },
    "Las Vegas Raiders": {
        "short": 1.08, "deep": 1.11, "te": 1.06, "rush": 1.08,
        "xpl": 1.08, "press": 0.93, "coverage_grade": 1.08
    },

    # MIXED / AVERAGE DEFENSES
    "Baltimore Ravens": {
        "short": 1.12, "deep": 1.15, "te": 1.08, "rush": 1.05,
        "xpl": 1.12, "press": 0.88, "coverage_grade": 1.12
    },
    "Kansas City Chiefs": {
        "short": 1.02, "deep": 1.05, "te": 1.01, "rush": 0.98,
        "xpl": 1.02, "press": 0.96, "coverage_grade": 1.02
    },
    "Tampa Bay Buccaneers": {
        "short": 1.04, "deep": 1.07, "te": 1.02, "rush": 0.96,
        "xpl": 1.04, "press": 0.94, "coverage_grade": 1.04
    },
    "Buffalo Bills": {
        "short": 0.94, "deep": 0.96, "te": 0.95, "rush": 1.22,
        "xpl": 0.93, "press": 1.05, "coverage_grade": 0.95
    },
    "New York Giants": {
        "short": 1.01, "deep": 1.03, "te": 1.02, "rush": 0.98,
        "xpl": 1.02, "press": 0.97, "coverage_grade": 1.02
    },

    # ELITE RUSH DEFENSES
    "San Francisco 49ers": {
        "short": 0.95, "deep": 0.93, "te": 0.94, "rush": 0.82,
        "xpl": 0.88, "press": 1.14, "coverage_grade": 0.94
    },
    "Cincinnati Bengals": {
        "short": 0.98, "deep": 0.96, "te": 0.97, "rush": 0.84,
        "xpl": 0.90, "press": 1.06, "coverage_grade": 0.97
    },

    # REMAINING TEAMS
    "Miami Dolphins": {
        "short": 0.99, "deep": 1.01, "te": 0.98, "rush": 1.05,
        "xpl": 0.98, "press": 0.98, "coverage_grade": 0.99
    },
    "Dallas Cowboys": {
        "short": 0.97, "deep": 0.95, "te": 0.96, "rush": 1.08,
        "xpl": 0.95, "press": 1.02, "coverage_grade": 0.96
    },
    "Minnesota Vikings": {
        "short": 1.03, "deep": 1.01, "te": 1.02, "rush": 1.06,
        "xpl": 1.01, "press": 0.97, "coverage_grade": 1.02
    },
    "Detroit Lions": {
        "short": 0.96, "deep": 0.98, "te": 0.97, "rush": 1.04,
        "xpl": 0.96, "press": 1.01, "coverage_grade": 0.97
    },
    "Atlanta Falcons": {
        "short": 1.06, "deep": 1.04, "te": 1.05, "rush": 1.10,
        "xpl": 1.05, "press": 0.95, "coverage_grade": 1.05
    },
    "New Orleans Saints": {
        "short": 0.93, "deep": 0.91, "te": 0.92, "rush": 1.02,
        "xpl": 0.90, "press": 1.08, "coverage_grade": 0.92
    },
    "Seattle Seahawks": {
        "short": 1.01, "deep": 1.03, "te": 1.00, "rush": 1.08,
        "xpl": 1.01, "press": 0.96, "coverage_grade": 1.01
    },
    "Los Angeles Rams": {
        "short": 0.98, "deep": 1.00, "te": 0.99, "rush": 1.06,
        "xpl": 0.98, "press": 0.99, "coverage_grade": 0.99
    },
    "Arizona Cardinals": {
        "short": 1.04, "deep": 1.02, "te": 1.03, "rush": 1.12,
        "xpl": 1.03, "press": 0.94, "coverage_grade": 1.03
    },
    "Washington Commanders": {
        "short": 1.02, "deep": 1.04, "te": 1.01, "rush": 1.09,
        "xpl": 1.02, "press": 0.96, "coverage_grade": 1.02
    }
}

# ---------------------------------------------------
# Helper functions
# ---------------------------------------------------

def get_enhanced_def_splits(team_name: str) -> dict:
    base_def = {
        "short": 1.0, "deep": 1.0, "te": 1.0, "rush": 1.0,
        "xpl": 1.0, "press": 1.0, "coverage_grade": 1.0
    }
    return DEF_ADJ_2025.get(team_name, base_def)

def get_def_split(team):
    adj = get_enhanced_def_splits(team)
    return (
        float(adj["short"]),
        float(adj["deep"]),
        float(adj["te"]),
        float(adj["rush"]),
        float(adj["xpl"])
    )

def get_press_factor(team):
    adj = get_enhanced_def_splits(team)
    return float(adj["press"])

def get_def_strength_tier(team_name: str) -> str:
    adj = get_enhanced_def_splits(team_name)
    pass_score = (adj["short"] + adj["deep"] + adj["te"]) / 3
    overall_score = (pass_score + adj["rush"]) / 2
    if overall_score <= 0.90:
        return "elite"
    elif overall_score <= 0.98:
        return "above_average"
    elif overall_score <= 1.05:
        return "average"
    elif overall_score <= 1.12:
        return "below_average"
    else:
        return "poor"

def update_def_adj_weekly(team_name: str, week_performance: dict):
    if team_name not in DEF_ADJ_2025:
        return
    current = DEF_ADJ_2025[team_name]
    if week_performance.get("pass_yards_allowed", 0) > 400:
        current["short"] = min(1.25, current["short"] * 1.02)
        current["deep"] = min(1.25, current["deep"] * 1.02)
        current["te"] = min(1.25, current["te"] * 1.02)
    if week_performance.get("sacks", 0) >= 4:
        current["press"] = max(0.80, current["press"] * 0.98)
    mean_regression = 0.95
    for key in current:
        if key != "coverage_grade":
            if current[key] > 1.0:
                current[key] = 1.0 + (current[key] - 1.0) * mean_regression
            else:
                current[key] = 1.0 - (1.0 - current[key]) * mean_regression

def get_situational_def_adj(team_name: str, situation: str) -> dict:
    base_adj = get_enhanced_def_splits(team_name)
    situational_mults = {
        'red_zone': {
            "Cleveland Browns": 0.85, "Philadelphia Eagles": 0.87,
            "New England Patriots": 1.15, "Jacksonville Jaguars": 1.12
        },
        'third_down': {
            "Green Bay Packers": 0.88, "Denver Broncos": 0.90,
            "Carolina Panthers": 1.10, "Tennessee Titans": 1.08
        },
        'two_minute': {
            "Philadelphia Eagles": 0.82, "Pittsburgh Steelers": 0.95,
            "New England Patriots": 1.18, "Las Vegas Raiders": 1.12
        }
    }
    mult = situational_mults.get(situation, {}).get(team_name, 1.0)
    return {k: v * mult for k, v in base_adj.items()}

def validate_def_adjustments():
    print("Defensive Adjustment Validation:")
    print("=" * 50)
    elite_defenses = []
    poor_defenses = []
    for team, adj in DEF_ADJ_2025.items():
        tier = get_def_strength_tier(team)
        avg_pass_adj = (adj["short"] + adj["deep"] + adj["te"]) / 3
        if tier == "elite":
            elite_defenses.append((team, avg_pass_adj))
        elif tier == "poor":
            poor_defenses.append((team, avg_pass_adj))
    print("Elite Defenses:")
    for team, adj in sorted(elite_defenses, key=lambda x: x[1]):
        print(f"  {team}: {adj:.3f}")
    print("\nPoor Defenses:")
    for team, adj in sorted(poor_defenses, key=lambda x: x[1], reverse=True):
        print(f"  {team}: {adj:.3f}")

if __name__ == "__main__":
    validate_def_adjustments()


# ---------------------------------------------------# Offensive Tendencies
def get_off_tend(team):
    d = OFF_TEND.get(team, {"pace": 1.0, "pass_rate": 1.0, "red_zone_aggression": 1.0})
    return float(d["pace"]), float(d["pass_rate"]), float(d.get("red_zone_aggression", 1.0))

# === Play-budget core (single source of truth) ===
BASE_PLAYS_PER_GAME = 62.0  # league mean non-OT

def estimate_play_budget(team_name: str) -> int:
    pace, _pass_mult, _ = get_off_tend(team_name)
    return int(round(BASE_PLAYS_PER_GAME * max(0.85, min(1.20, pace))))

def expected_dropbacks(qb_params: dict, def_team: str) -> tuple[float,float,float]:
    """
    Return (attempts, sacks, total_dropbacks).
    IMPORTANT: qb_params['att_mu'] is treated as *attempts* after scrambles.
    We re-inflate to dropbacks using sack rate so plays are conserved.
    """
    att = float(qb_params["att_mu"])
    sr  = sack_rate_per_dropback(qb_params["team"], def_team, qb_params["name"])
    # dropbacks = att + sacks, with sacks = sr * dropbacks
    # => dropbacks = att / (1 - sr)
    dropbacks = att / max(0.50, (1.0 - sr))  # clamp to avoid divide-by-small
    sacks = dropbacks - att
    return att, sacks, dropbacks



# ---------------------------------------------------
# Utility
# ---------------------------------------------------
def _f(x, default):
    try:
        return float(x)
    except Exception:
        return float(default)

def truncated_lognormal(sigma, scale, upper_cap):
    y = lognorm.rvs(sigma, scale=scale)
    return min(y, upper_cap)

# ---------------------------------------------------
# Param builders (use opponent split def + press)
# ---------------------------------------------------

def build_qb_params(player, team_name: str, opponent_name: str, opp_split, press_factor: float):
    # accept dict (from get_enhanced_def_splits) or tuple (from get_def_split)
    if isinstance(opp_split, dict):
        short_adj, deep_adj = float(opp_split["short"]), float(opp_split["deep"])
    else:
        short_adj, deep_adj, _te_adj, _rush_adj, _xpl_adj = opp_split

    stats = player.get("stats", {})
    name = player["athlete"]["displayName"]
    style = QB_STYLE_CANON.get(canonical_name(name), {})
    pace, pass_rate, _rza = get_off_tend(team_name)

    # Attempts
    att_mu_base = style.get("att_mu", _f(stats.get("passingAttempts", 30), 30))
    att_mu = att_mu_base * pace * pass_rate

    # Completion %
    raw_comp = style.get("comp_p", _f(stats.get("completionPct", 0.65), 0.65))
    coverage = get_enhanced_def_splits(opponent_name).get("coverage_grade", 1.0)
    comp_p = raw_comp / (press_factor ** 0.06)
    comp_p *= (1.05 - (coverage - 1.0))
    comp_p = min(max(comp_p, 0.45), 0.80)

    # YPA
    base_ypa = style.get("ypa_mu_eff", _f(stats.get("yardsPerPass", 7.5), 7.5)) + style.get("ypa_shift", 0.0)
    if name in {"Daniel Jones", "Bo Nix"}:
        base_ypa += 0.8

    short_share = 0.75 if name == "Tua Tagovailoa" else 0.65
    deep_share = 1.0 - short_share
    ypa_mu_eff = base_ypa * (short_adj * short_share + deep_adj * deep_share)

    if name == "Tua Tagovailoa":
        cap = style.get("ypa_cap", 6.8)
        tier = get_def_strength_tier(opponent_name)
        if tier == "elite" and press_factor > 1.03:
            cap = min(cap, style.get("elite_cap", 6.6))
        ypa_mu_eff = min(ypa_mu_eff, cap)

    rush_att_base = max(_f(stats.get("rushingAttempts", 3), 3), style.get("rush_att_floor", 0))
    qb_rush_boost = style.get("qb_rush_boost", 1.0)

    return {
        "name": name, "team": team_name,
        "att_mu": att_mu, "att_k": 10,
        "comp_p": comp_p,
        "ypa_mu_eff": max(ypa_mu_eff, 5.0),
        "ypa_sigma": 0.25 if name != "Tua Tagovailoa" else QB_STYLE["Tua Tagovailoa"].get("trunc_sigma", 0.22),
        "sack_lambda": 2.0 * press_factor * style.get("sack_lambda_boost", 1.0),
        "rz_mult": style.get("rz_mult", 1.0),
        "rush_att_lambda": rush_att_base * qb_rush_boost * pace,
        "scramble_share": style.get("scramble_share", 0.30),
        "scr_shape": style.get("scramble_shape", 3.8), "scr_scale": style.get("scramble_scale", 1.4),
        "des_shape": style.get("designed_shape", 3.6), "des_scale": style.get("designed_scale", 1.2),
        "scr_press_mult": style.get("scramble_extra_press_mult", 0.10),
        "trailing_bonus": style.get("trailing_bonus", 0.08 if name != "Tua Tagovailoa" else 0.03),
    }


def build_receivers(players, team_name, qb_name, rush_adj, te_adj, short_adj, xpl_adj, opp_press=1.0):
    """Build receiver parameters for simulation"""
    receivers = []

    # Get target weights from overrides
    target_weights = TARGET_WEIGHT_OVERRIDES_2025_2026.get(team_name, {})

    for player in players:
        name = player["athlete"]["displayName"]
        name_can = canonical_name(name)
        pos = get_position_abbr(player["athlete"])

        if pos in ["WR", "TE"]:
            rush_att_lambda = 0.5  # Occasional jet sweeps/trick plays
        elif pos in ["RB", "FB"]:
            rush_att_lambda = get_rb_baseline_2025(name, team_name).get("rush_att_lambda", 8.0)
        else:
            rush_att_lambda = 0.1
        # Default weight or from overrides (match canonically)
        weight = target_weights.get(name_can, 1.0 / max(1, len(players)))

        # Position-based adjustments
        if pos == "TE":
            catch_p = 0.72
            ypt_mu = 11.5 / te_adj

            if name in ["Mark Andrews", "Travis Kelce", "Sam LaPorta"]:
              pressure_boost = 1.0 + max(0.0, (opp_press - 1.0) * 0.25)
              weight *= pressure_boost

            if name == "Mark Andrews":
                weight *= 1.15

        elif pos in ["RB", "FB"]:
            catch_p = 0.68
            ypt_mu = 8.5 / short_adj
            weight *= (1.0 + 0.1 * max(0, 1.0 - rush_adj))
        else:  # WR
            catch_p = 0.65
            ypt_mu = 13.0 / ((short_adj + xpl_adj) / 2)

        receivers.append({
            "name": name,
            "pos": pos,
            "weight": weight,
            "catch_p": min(0.82, catch_p),
            "ypt_mu": max(5.5, ypt_mu),
            "ypt_sigma": 0.35,
            "rush_att_lambda": 0.2 if pos in ["WR", "TE"] else get_rb_baseline_2025(name, team_name).get("rush_att_lambda", 1.0),
            "rush_shape": get_rb_baseline_2025(name, team_name).get("rush_shape", 4.0),
            "rush_scale": get_rb_baseline_2025(name, team_name).get("rush_scale", 1.0),
            "explosive_p": 0.15,
            "expl_mu": 1.8,
            "expl_sigma": 0.5
        })

    return receivers
# ---------------------------------------------------
# Scoring variance helpers (DROP BEFORE script)
# ---------------------------------------------------
def compute_pass_tds(att, rz_mult=1.0, state="neutral", shootout=False):
    """
    NegBin TD model with a shootout lift; replace Poisson TD draws with this.
    att: team/ QB attempts in sim.
    """
    lam = (att / 27.0) * 1.70 * rz_mult
    if state == "trailing":
        lam *= 1.12
    if shootout:
        lam *= 1.35  # fat tail
    # NegBin(r, p) with mean=lam and overdispersion
    r = 2.2
    p = r / (r + max(lam, 1e-6))
    return int(nbinom.rvs(r, p))

def fg_attempts_mean(base=1.05):
    """
    Slightly higher/variable FG volume to capture FG-decided finales.
    """
    spike = 1 + (np.random.rand() < 0.20) * np.random.choice([1, 2])
    return max(0.0, base * spike)

# ---------------------------------------------------
# Direct game-flow engine (per-sim state machine)
# ---------------------------------------------------
SCRIPT = {
    "leading": {"pass_mult":0.92, "rush_mult":1.10, "rb_bias": +0.10, "wr2_bias": -0.06},
    "neutral": {"pass_mult":1.00, "rush_mult":1.00, "rb_bias": 0.00, "wr2_bias": 0.00},
    "trailing":{"pass_mult":1.12, "rush_mult":0.90, "rb_bias": -0.08, "wr2_bias": +0.08},
}
SCRIPT_PRIORS = np.array([0.28, 0.44, 0.28])  # leading, neutral, trailing

def choose_state(score_diff):
    if score_diff >= 7:  return "leading"
    if score_diff <= -7: return "trailing"
    return "neutral"

def allocate_targets(team_attempts, weights):
    weights=np.array(weights, dtype=float)
    if weights.sum()<=0: weights=np.ones_like(weights)/len(weights)
    shares=dirichlet(weights*28)
    return np.random.multinomial(max(int(team_attempts),0), shares), shares

def allocate_pass_tds_to_receivers(qb_pass_tds, player_names, team_name):
    """
    Use team RZ weights if present; clamp to players on field; otherwise uniform.
    Returns list of TD counts aligned to player_names.
    """
    if qb_pass_tds <= 0 or not player_names:
        return [0] * len(player_names)

    rz = RZ_WEIGHTS_CANON.get(team_name, {})


    present = set(player_names)

    if rz:
        w = np.array([rz.get(n, 0.0) for n in player_names], dtype=float)
        # if all zeros (weights missing for these specific players), fallback to uniform
        if w.sum() <= 0:
            w = np.ones(len(player_names), dtype=float)
    else:
        w = np.ones(len(player_names), dtype=float)

    # small smoothing so guys with tiny weights can still spike
    w = w + 0.01
    w = w / w.sum()

    # multinomial allocate TDs
    td_alloc = np.random.multinomial(qb_pass_tds, w)
    return td_alloc.tolist()

##

def simulate_qb_and_receivers(qb_params, receivers, opp_team_name, opp_splits, press_factor, roster, n_sims=5000):
    short_adj, deep_adj, te_adj, rush_adj, xpl_adj = opp_splits
    qb_name = qb_params["name"]
    off_t   = qb_params.get("team_name") or qb_params.get("team")
    def_t   = qb_params.get("opp_team_name", opp_team_name)
    mobile_qb = qb_name in MOBILE_QB_SET

    names   = [p["name"] for p in receivers]
    weights = np.array([p["weight"] for p in receivers], dtype=float)
    catch_p = np.array([np.clip(p["catch_p"], 0.40, 0.82) for p in receivers], dtype=float)
    ypt_mu  = np.array([max(p["ypt_mu"], 5.5) for p in receivers], dtype=float)
    ypt_sigma = np.array([p["ypt_sigma"] for p in receivers], dtype=float)

    # --- NEW: lock WR/TE rushing to the tiny WR-jet budget ---
    # pull everything we need from existing args so we don't depend on earlier locals
    qb_name = qb_params.get("name", "")
    off_t   = qb_params.get("team", "")
    def_t   = opp_team_name
    local_att_mu = int(round(qb_params.get("att_mu", 32)))

    # build budget and take only the WR gadget slice
    budget    = derive_rush_budget(off_t, {"name": qb_name, "team": off_t, "att_mu": local_att_mu}, def_t)
    wr_total  = int(budget.get("wr_rushes", 0))

    # allocate those few carries to specific WRs (or 0 if no weights/team mapping)
    jet_weights = WR_RUSH_WEIGHTS.get(off_t, {})
    jet_by_name = {n: 0 for n in names}

    if wr_total > 0 and jet_weights:
        w = np.array([jet_weights.get(n, jet_weights.get("Other", 0.0)) for n in names], dtype=float)
        s = float(w.sum())
        if s > 0.0:
            w = w / s
            alloc = np.floor(w * wr_total).astype(int)
            rem = wr_total - int(alloc.sum())
            if rem > 0:
                frac = (w * wr_total) - alloc
                for idx in np.argsort(-frac)[:rem]:
                    alloc[idx] += 1
            for i, n in enumerate(names):
                jet_by_name[n] = int(alloc[i])
    else:
        # leave jet_by_name all zeros if no allowed jets for this team
        pass

    # FINAL: receivers’ rush attempts are ONLY their allocated jet sweeps
    # (everyone else gets 0; no phantom 6–10 carries anymore)

    # --- Build the rush allocation map once ---
    rush_info = derive_rush_budget(off_t, qb_params, opp_team_name)
    rb_alloc  = allocate_rb_carries(off_t, qb_params["name"], opp_team_name, rush_info["rb_carries_total"])
    rush_by_player = assign_rush_attempts_to_players(off_t, qb_params["name"], roster, rush_info, rb_alloc)




    r_att = np.array([rush_by_player.get(canonical_name(n), 0) for n in names], dtype=int)





    r_shape = np.array([p["rush_shape"] for p in receivers], dtype=float)
    r_scale = np.array([p["rush_scale"] for p in receivers], dtype=float) * (RB_SYNERGY_YPC_BOOST if mobile_qb else 1.0)

    expl_p     = np.array([p["explosive_p"] for p in receivers], dtype=float)
    expl_mu    = np.array([p["expl_mu"] for p in receivers], dtype=float)
    expl_sigma = np.array([p["expl_sigma"] for p in receivers], dtype=float)

    qb_out = {"Attempts":[], "Completions":[], "PassYards":[], "PassTDs":[], "INTs":[], "Sacks":[],
              "RushAtt":[], "RushYards":[], "RushTDs":[]}
    rec_out = {n: {"Targets":[], "Receptions":[], "RecYards":[], "RecTDs":[],
                   "RushAtt":[], "RushYards":[], "RushTDs":[]} for n in names}

    states = np.random.choice(["leading","neutral","trailing"], size=n_sims, p=SCRIPT_PRIORS)

    rb_allocations_by_script = {}
    for script in ["leading", "neutral", "trailing"]:
      budget = derive_rush_budget(off_t, qb_params, opp_team_name)
      rb_allocations_by_script[script] = allocate_rb_carries(
          off_t, qb_params["name"], opp_team_name,
          budget["rb_carries_total"], script  # Pass game script here
      )



    for _ in range(n_sims):
        state = states[_]
        d = SCRIPT[state]
        current_rb_alloc = rb_allocations_by_script.get(state, {})


        # 1) Attempts first (so attempts exists before sacks)
        att_mu = qb_params["att_mu"] * d["pass_mult"]
        attempts = int(nbinom.rvs(qb_params['att_k'], qb_params['att_k']/(qb_params['att_k'] + max(att_mu, 0.1))))

        # 2) YPA sample → yards
        base_ypa = qb_params["ypa_mu_eff"]
        mu = np.log(max(base_ypa, 0.1)) - 0.5*qb_params['ypa_sigma']**2
        if qb_name == "Tua Tagovailoa":
            ypa = truncated_lognormal(qb_params['ypa_sigma'], np.exp(mu), upper_cap=8.0)
            trailing_bonus = qb_params.get("trailing_bonus", 0.03)
        else:
            ypa = lognorm.rvs(qb_params['ypa_sigma'], scale=np.exp(mu))
            trailing_bonus = qb_params.get("trailing_bonus", 0.08)

        pass_yards = attempts * ypa * (1.0 + (trailing_bonus if state=="trailing" else 0.0))
        completions = np.random.binomial(max(attempts,0), np.clip(qb_params['comp_p'],0.35,0.82))

        # 3) Sacks computed AFTER attempts are known
        dropbacks = max(1, attempts)
        s_p = sack_rate_per_dropback(off_t, def_t, qb_name)
        sacks = np.random.binomial(n=dropbacks, p=s_p)
        sack_yards_lost = 0.0 if sacks == 0 else float(np.random.gamma(shape=3.2, scale=1.8, size=sacks).sum())
        pass_yards = max(0.0, pass_yards - 0.7 * sack_yards_lost)

        # 4) QB rushing
        base_rush_att = qb_params['rush_att_lambda'] * d["rush_mult"]
        extra_scrambles = 0
        if qb_name in QB_STYLE:
            extra_scrambles = poisson.rvs(max(0.0, (attempts/30.0) * max(0.0, press_factor-1.0) * QB_STYLE[qb_name].get("scramble_extra_press_mult", 0.10)))
        total_rush_att = max(0, poisson.rvs(max(base_rush_att,0.0))) + extra_scrambles
        scramble_share = qb_params.get("scramble_share", 0.30)
        scr = np.random.binomial(total_rush_att, np.clip(scramble_share,0.0,1.0))
        des = max(0, total_rush_att - scr)

        scr_ypc = gamma.rvs(qb_params.get("scr_shape",4.0), scale=qb_params.get("scr_scale",1.5) * rush_adj)
        des_ypc = gamma.rvs(qb_params.get("des_shape",3.6), scale=qb_params.get("des_scale",1.2) * rush_adj)
        rush_yards = scr*scr_ypc + des*des_ypc + (lognorm.rvs(0.50, scale=np.exp(2.0)) if np.random.rand()<0.22 else 0.0)

        rush_td_boost = QB_STYLE.get(qb_name, {}).get("rush_td_boost", 1.0)
        rush_tds = poisson.rvs((total_rush_att/11.0) * 0.42 * rush_adj * rush_td_boost)

        # 5) Passing TDs / INTs
        wr1_boost = 1.05 if max(weights) > 0.25 else 1.0
        pass_tds = poisson.rvs((attempts/28.0) * 1.65 * qb_params.get("rz_mult", 1.0) * wr1_boost)

        _, _, own_turn_add, _ = get_penalty_factors(off_t)
        _, _, opp_turn_add, _ = get_penalty_factors(def_t)
        pen_turn_adj = own_turn_add - (opp_turn_add * 0.35)
        ints = poisson.rvs((attempts/30.0) * (1.10 - min(short_adj,deep_adj)*0.6) * press_factor * (1.0 + min(0.20, pen_turn_adj)))
        if np.random.rand() < max(0.0, min(0.22, 0.75*pen_turn_adj)):
            if np.random.rand() < 0.55:
                ints += 1
            else:
                pass_yards = max(0.0, pass_yards - 0.5 * float(np.random.gamma(2.2, 2.0)))
                sacks += 1
        if state == "trailing" and np.random.rand() < 0.20:
            if np.random.rand() < 0.5: ints += 1
            else: pass_tds += 1

        if state == "trailing" and press_factor > 1.06 and np.random.rand() < 0.25:
            extra_scrambles += 1


        # 6) Target allocation
        bias_weights = weights.copy()
        for j in range(len(names)):
            bias_weights[j] = max(0.01, bias_weights[j] + (d["rb_bias"] if r_att[j] > 0 else d["wr2_bias"]))
        bias_weights = np.clip(bias_weights, 0.01, None)
        bias_weights /= bias_weights.sum()

        tgt_vec, share_vec = allocate_targets(attempts, bias_weights)

        # 7) Receiver outcomes
        recs = np.zeros(len(names), dtype=int)
        rec_yds = np.zeros(len(names), dtype=float)
        for j in range(len(names)):
            t = int(tgt_vec[j])
            r = np.random.binomial(t, catch_p[j]) if t>0 else 0
            mu_r = np.log(max(ypt_mu[j]*short_adj, 5.2)) - 0.5*ypt_sigma[j]**2
            ypt = lognorm.rvs(ypt_sigma[j], scale=np.exp(mu_r))
            recs[j] = r
            rec_yds[j] = r * ypt

        td_alloc = allocate_pass_tds_to_receivers(pass_tds, names, off_t)

        # >>> vectorized per-receiver rush attempts/yards (fixes patt being scalar)


        script_adjusted_att = np.zeros_like(r_att)
        for j, name in enumerate(names):
            canonical_name_j = canonical_name(name)
            script_carries = current_rb_alloc.get(name, current_rb_alloc.get(canonical_name_j, 0))
            if script_carries > 0:
                script_adjusted_att[j] = script_carries
            else:
                script_adjusted_att[j] = r_att[j]

        patt = np.random.poisson(np.clip(script_adjusted_att * d["rush_mult"], 0, None))




        pypc = gamma.rvs(np.clip(r_shape, 0.1, None), scale=np.clip(r_scale, 0.05, None), size=len(names))
        prush_yards = patt * pypc

        # fill outputs
        qb_out["Attempts"].append(attempts)
        qb_out["Completions"].append(completions)
        qb_out["PassYards"].append(pass_yards)
        qb_out["PassTDs"].append(pass_tds)
        qb_out["INTs"].append(ints)
        qb_out["Sacks"].append(sacks)
        qb_out["RushAtt"].append(total_rush_att)
        qb_out["RushYards"].append(rush_yards)
        qb_out["RushTDs"].append(rush_tds)

        for j, n in enumerate(names):
            rec_out[n]["Targets"].append(int(tgt_vec[j]))
            rec_out[n]["Receptions"].append(int(recs[j]))
            rec_out[n]["RecYards"].append(float(rec_yds[j]))
            rec_out[n]["RecTDs"].append(int(td_alloc[j]))
            rec_out[n]["RushAtt"].append(int(patt[j]))
            rec_out[n]["RushYards"].append(float(prush_yards[j]))
            rec_out[n]["RushTDs"].append(int(poisson.rvs(patt[j] / 18.0)))
    # convert to DataFrames (see fix #1)
    qb_df = pd.DataFrame(qb_out)
    rec_dfs = {n: pd.DataFrame(rec_out[n]) for n in names}
    return qb_df, rec_dfs


# ---------------------------------------------------
# Summaries & Team scoring (no deprecated option_context)
# ---------------------------------------------------
def summarize(df):
    if isinstance(df, dict):
        df = pd.DataFrame(df)
    clean = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    desc = clean.describe(percentiles=[0.25, 0.5, 0.75]).T
    return desc[["25%", "50%", "75%"]]


def simulate_team_score(team_name: str, qb_df: pd.DataFrame, recv_dfs: dict[str, pd.DataFrame]):
    scores = []
    fg_means = []
    n = len(qb_df)
    pen_count, pen_yards, _, yard_impact = get_penalty_factors(team_name)

    for i in range(n):
        td = int(qb_df["RushTDs"].iloc[i])
        yards = float(qb_df["PassYards"].iloc[i] + qb_df["RushYards"].iloc[i])

        for df in recv_dfs.values():
            td += int(df["RecTDs"].iloc[i] + df["RushTDs"].iloc[i])
            yards += float(df["RecYards"].iloc[i] + df["RushYards"].iloc[i])

        td_pts = td * 7

        # calibrated FG model with penalty impact
        fg_mean = 1.10 / (0.25*yard_impact + 0.75)
        fg_means.append(fg_mean)
        fg = np.random.poisson(max(fg_mean, 0.0))

        # end-game swing always defined
        swing = np.random.choice([-6, -3, 0, 3, 6], p=[0.1, 0.2, 0.4, 0.2, 0.1])
        if pen_count >= 8 and np.random.rand() < 0.12:
            swing += np.random.choice([-6, -3, 3, 6], p=[0.2, 0.3, 0.3, 0.2])

        score = td_pts + fg*3 + swing
        scores.append(score)

    s = pd.Series(scores)
    return (s.median(), s.quantile(0.25), s.quantile(0.75), float(np.mean(fg_means)))


# ---------------------------------------------------
# Driver
# ---------------------------------------------------


# --- add this helper anywhere above run_matchup_by_name ---
# --- helper, place above run_matchup_by_name ---
# duplicate helper removed — using the robust implementation defined earlier

# duplicate filter_players_on_roster removed — using the canonical implementation above

def sanitize_focus_pool(team: str, roster: list[dict]) -> list[dict]:
    focus = FOCUS_PLAYERS.get(team, [])
    if not focus:
        return roster[:10]
    return filter_players_on_roster(team, roster, focus)




def run_one_side(off_name: str, def_name: str, n_sims: int):
    # defense context for this side
    splits = get_def_split(def_name)
    press  = get_press_factor(def_name)

    tid = fetch_team_id(off_name)
    roster = normalize_roster(fetch_team_roster(tid))
    roster = filter_active_roster(roster)

    # build focus players but ensure they exist on the fetched roster
    focus = FOCUS_PLAYERS.get(off_name, [])
    roster_index = build_roster_index(roster)
    focus_on_roster = [roster_index[canonical_name(n)] for n in focus if canonical_name(n) in roster_index]
    players = focus_on_roster or roster[:10]

    qb = pick_qb_from_roster(off_name, roster)
    qb_params = build_qb_params(
        qb, off_name, def_name,
        {"short": splits[0], "deep": splits[1], "te": splits[2], "rush": splits[3], "xpl": splits[4]},
        press
    )
    qb_params["team_name"] = off_name
    qb_params["opp_team_name"] = def_name

    recv_pool = [p for p in players if p["athlete"]["id"] != qb["athlete"]["id"]]
    if not recv_pool:
        recv_pool = [p for p in roster if p["athlete"]["id"] != qb["athlete"]["id"]][:8]

    recv_params = build_receivers(
        recv_pool, off_name, qb_params["name"],
        splits[3], splits[2], splits[0], splits[4],
        opp_press=press
    )

    qb_df, rec_dfs = simulate_qb_and_receivers(qb_params, recv_params, def_name, splits, press, roster, n_sims=n_sims)
    return qb, splits, press, qb_params, qb_df, rec_dfs

def run_matchup_by_name(team1, team2, n_sims=3500):
    # --- team1 offense vs team2 defense ---
    qb1, splits1, press1, qb_params1, qb_df1, rec_dfs1 = run_one_side(team1, team2, n_sims)
    # --- team2 offense vs team1 defense ---
    qb2, splits2, press2, qb_params2, qb_df2, rec_dfs2 = run_one_side(team2, team1, n_sims)

    # print summaries
    for off_name, def_name, splits, press, qb, qb_df, rec_dfs in [
        (team1, team2, splits1, press1, qb1, qb_df1, rec_dfs1),
        (team2, team1, splits2, press2, qb2, qb_df2, rec_dfs2),
    ]:
        print(f"\n==============================\n📊 {off_name} offense vs {def_name} defense")
        print(f"(DEF split → short {splits[0]:.2f}, deep {splits[1]:.2f}, TE {splits[2]:.2f}, rush {splits[3]:.2f}, xpl {splits[4]:.2f}; press {press:.2f})")
        pace, pr, rza = get_off_tend(off_name)
        print(f"(OFF tend → pace {pace:.2f}, passrate {pr:.2f}, RZ aggr {rza:.2f})")
        print("==============================")
        print(f"\n--- {qb['athlete']['displayName']} (QB) ---")
        print(summarize(qb_df).round(2))
        for name, df in rec_dfs.items():
            print(f"\n--- {name} ---")
            print(summarize(df).round(2))
        score = simulate_team_score(off_name, qb_df, rec_dfs)
        print(f"\n🏆 {off_name} Score (median [25–75]): {score[0]:.1f}  ({score[1]:.1f}–{score[2]:.1f}) | avg FG mean {score[3]:.2f}")

    return {
        "team1": {"qb_df": qb_df1, "rec_dfs": rec_dfs1},
        "team2": {"qb_df": qb_df2, "rec_dfs": rec_dfs2},
    }



import io
import sys

def run_and_capture(team1, team2, n_sims=5000):
    """
    Capture printed output of run_matchup_by_name into a string
    so we can save it to a file instead of only printing.
    """
    buffer = io.StringIO()
    sys_stdout = sys.stdout
    sys.stdout = buffer
    try:
        run_matchup_by_name(team1, team2, n_sims=n_sims)
    finally:
        sys.stdout = sys_stdout
    return buffer.getvalue()

def main():
    parser = argparse.ArgumentParser(description="NFL1TOMILLIONS — Direct Game-Flow Sim (Monolith)")
    parser.add_argument("team1", nargs="?", default="Buffalo Bills")
    parser.add_argument("team2", nargs="?", default="Miami Dolphins")
    parser.add_argument("--sims", type=int, default=5000)
    parser.add_argument("--out", type=str, default="sim_results.txt", help="Output file for summary stats")
    args, _ = parser.parse_known_args()

    # capture printed summary stats
    summary = run_and_capture(args.team1, args.team2, n_sims=args.sims)

    # write to file
    with open(args.out, "w", encoding="utf-8") as f:
        f.write(summary)

    print(f"\n✅ Summary written to {args.out}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("team1", nargs="?", default="Los Angeles Rams")
    parser.add_argument("team2", nargs="?", default="Tennessee Titans")
    parser.add_argument("--sims", type=int, default=3500)
    parser.add_argument("--out", type=str, default="sim_results.txt")
    args, _ = parser.parse_known_args()

    summary = run_and_capture("Kansas City Chiefs", "New York Giants", n_sims=15000)

    with open(args.out, "w", encoding="utf-8") as f:
        f.write(summary)

    print(f"\n✅ Summary written to {args.out}")


Target Share Validation:
Issues found:
  New York Giants: 0.910
  New York Giants - Tyrone Tracy Jr: 3.0% (too low)
  Washington Commanders - Austin Ekeler: 0.0% (too low)

Total teams: 32/32
RB Baseline Validation:
Elite RBs (18+ attempts):
  Saquon Barkley: 22.0
  Jonathan Taylor: 21.0
  Derrick Henry: 20.0
  Josh Jacobs: 19.5
  Bijan Robinson: 19.0
  Christian McCaffrey: 18.0
  Quinshon Judkins: 18.0

Committee/Backup RBs (≤10 attempts):
  Najee Harris: 0.0
  Joe Mixon: 0.0
  Aaron Jones: 0.0
  Austin Ekeler: 0.0
  Tyrone Tracy Jr.: 0.0
  James Conner: 0.0
  Ray Davis: 6.0
  Kareem Hunt: 6.5
  Tank Bigsby: 7.0
  David Montgomery: 8.0
  Justice Hill: 8.0
  Bucky Irving: 8.0
  Tyjae Spears: 9.0
  Brian Robinson Jr.: 10.0
  Jacory Croskey-Merritt: 10.0
  Ezekiel Elliott: 10.0

Total RBs covered: 45
Teams with depth charts: 32/32
Defensive Adjustment Validation:
Elite Defenses:
  Philadelphia Eagles: 0.837
  Cleveland Browns: 0.847
  Green Bay Packers: 0.877
  New York Jets: 0.907
  San